# Stage 01_mechanism_attribution: mechanism_attribution
Configuration, inputs and results remain in the corresponding stages directory.


### Initialization

Run the following setup cells from the Decoders working directory in the qec environment. They load this notebook's configuration, paths and dependencies; subsequent experiments reuse these definitions and should be read in execution order.


In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
mechanism_attribution_experiment_dir = ROOT / 'experiment/research/gap_mechanisms/stages/01_mechanism_attribution'
sys.path[:0] = [str(ROOT / 'src'), str(ROOT), str(ROOT / 'experiment')]

from experiment.utils.data_paths import ExperimentPaths

mechanism_attribution_data_paths = ExperimentPaths(ROOT, mechanism_attribution_experiment_dir / 'config/paths.json')


# BPOSD–Tesseract gap mechanism attribution experiments

Methods receive the same physical shots. Correction-based success requires a syndrome-valid correction in the sampled true logical sector; Tesseract predictions are compared directly with the truth. Instrumented BP uses this stage's `trace_bp`, not CythonBPOSD. Standard BPOSD returns the first syndrome-valid hard decision, or runs configured OSD on raw signed endpoint LLRs when BP does not converge. Figures are displayed inline; case data and aggregate statistics remain separate.


In [ ]:
from dataclasses import dataclass, replace
from itertools import combinations, product
from math import comb
from pathlib import Path
from time import perf_counter
import json
import sys
import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
import seaborn as sns
from scipy.optimize import Bounds, LinearConstraint, milp
from scipy.sparse import csc_matrix
from scipy.stats import wilcoxon
from tqdm.auto import tqdm
from beliefmatching import detector_error_model_to_check_matrices
from codes.ca_code import CA_9_2_5, TriangularCode
from decoders.cpp_backend import Tesseract
from decoders.cython.cython_core.osd_core import ordered_statistics_postprocess
from noises_ca import CircuitBuilder, ErrorRate

mechanism_attribution_config = json.loads((mechanism_attribution_experiment_dir / 'config/bposd_tesseract_gap_mechanism_attribution_experiments.json').read_text(encoding='utf-8'))
if mechanism_attribution_config['smoke_mode']:
    mechanism_attribution_config.update(
        primary_levels=[2],
        validation_levels=[3],
        p_values=[0.05],
        validation_p_values=[0.05],
        minimum_shots_per_point=60,
        maximum_shots_per_point=120,
        sampling_batch_size=60,
        target_gap_cases_per_point=1,
        analysis_gap_limit_per_point=2,
        bp_standard_iterations=8,
        continued_iterations=12,
        configured_full_osd_order_by_distance={'5': 2, '9': 2},
        pcm_views=3,
        selected_pcm_views=4,
        selected_pcm_case_limit=1,
        pcm_rows_per_view=[1],
        candidate_max_row_weight=8,
        restart_count=4,
        nms_alpha_values=[0.8, 1.0],
        oms_beta_values=[0.25],
        scaling_values=[0.8, 1.0],
        damping_values=[0.0, 0.5],
        prior_decode_multipliers=[0.9, 1.0, 1.1],
        prior_perturbations=3,
        pcm_ensemble_sizes=[2, 4],
        adaptive_max_views=2,
        logical_cost_case_limit_per_group=2,
        bootstrap_repetitions=50,
        final_case_limit_per_point=12,
        milp_time_limit_seconds=15.0,
        tesseract_timing_shots_per_batch=12,
    )

# Expected configuration values:
# print_level: integer; 1 = silent, 2 = key information, 3 = debug.
# primary_levels, validation_levels: values restricted to {2, 3}.
# primary_levels: non-empty.
# validation_levels: non-empty.
# p_values, validation_p_values: each item is in (0, 1).
# Expected relationship: minimum_shots_per_point <= maximum_shots_per_point.
# Expected relationship: continued_iterations > bp_standard_iterations.
# osd_orders: [0, 1, 2].
# pcm_views: positive.
# Expected relationship: selected_pcm_views >= pcm_views.
# sampling_batch_size: positive.
# target_gap_cases_per_point: positive.
# analysis_gap_limit_per_point: positive.
# bootstrap_repetitions: positive.
# restart_count: positive.
# ambiguity_top_k: positive.
# restart_message_sigma: non-negative.
# restart_prior_sigma: non-negative.
# prior_variable_sigma: non-negative.
# nms_alpha_values, scaling_values: each item is positive.
# oms_beta_values: each item is non-negative.
# damping_values: each item is in [0, 1).
# configured_full_osd_order_by_distance: keys exactly {'5', '9'}.
# values of configured_full_osd_order_by_distance: each item is at least 2.
# tesseract_beam: positive.
# tesseract_pq_limit: positive.
# tesseract_timing_shots_per_batch: positive.
# final_case_limit_per_point: -1 or positive.
# development_fraction: in (0, 1).
mechanism_attribution_name = 'bposd_tesseract_gap_mechanism_attribution_smoke' if mechanism_attribution_config['smoke_mode'] else 'bposd_tesseract_gap_mechanism_attribution'
mechanism_attribution_out = mechanism_attribution_data_paths.run(smoke_mode=mechanism_attribution_config['smoke_mode'])

from decoders.cython.cython_core.osd_core import OSD_VERSION, osd_candidate_count
from experiment.utils.misc import check_algorithm_cache
check_algorithm_cache(mechanism_attribution_data_paths, mechanism_attribution_config, {'osd': OSD_VERSION})

from experiment.utils.bp_trace import check_bp_cache
check_bp_cache(mechanism_attribution_data_paths, mechanism_attribution_config)

mechanism_attribution_conf, mechanism_attribution_trace_dir, mechanism_attribution_stats = (mechanism_attribution_data_paths.output('config'), mechanism_attribution_data_paths.output('trace'), mechanism_attribution_data_paths.output('stats'))

for mechanism_attribution_directory in [mechanism_attribution_conf, mechanism_attribution_trace_dir, mechanism_attribution_stats]:
    mechanism_attribution_directory.mkdir(parents=True, exist_ok=True)

mechanism_attribution_config_path = mechanism_attribution_data_paths.output('config_json')
if not mechanism_attribution_config_path.is_file() or not mechanism_attribution_config['reuse_existing_data']:
    mechanism_attribution_config_path.write_text(json.dumps({'config': mechanism_attribution_config}, indent=2), encoding='utf-8')

from experiment.utils import misc

def mechanism_attribution_log(level, payload):
    return misc.log(level, payload, CONFIG=mechanism_attribution_config)

def mechanism_attribution_stage_cached(paths):
    paths = [Path(path) for path in paths]
    present = [path.is_file() for path in paths]
    assert all(present) or not any(present), f'partial cache: {[str(p) for p, ok in zip(paths, present) if ok]}'

    return mechanism_attribution_config['reuse_existing_data'] and all(present)

mechanism_attribution_bit_columns = ['syndrome', 'truth', 'error', 'correction', 'hard', 'logical_sector', 'standard_sector', 'tesseract_sector', 'true_sector']

def mechanism_attribution_read_csv(path):
    header = pd.read_csv(path, nrows=0).columns
    dtype = {column: str for column in mechanism_attribution_bit_columns + ['shot_id'] if column in header}
    return pd.read_csv(path, dtype=dtype, low_memory=False)

mechanism_attribution_save_csv = misc.save_csv
mechanism_attribution_bits = misc.bits
mechanism_attribution_bit_array = misc.bit_array_variant_3

from experiment.utils import gap_mechanisms

def mechanism_attribution_split_for(shot_id):
    return gap_mechanisms.split_for(shot_id, CONFIG=mechanism_attribution_config)

sns.set_theme(style='whitegrid')
mechanism_attribution_log(2, {'output': str(mechanism_attribution_out)})


## Instrumented BP, OSD and exact logical-sector primitives

Define shared decoding and diagnostic operations so the subsequent interventions retain the same syndrome, priors and logical labels. Exact logical-sector quantities are offline measurements for the tractable code.


In [ ]:
from experiment.utils import gap_mechanisms

mechanism_attribution_CodeContext = gap_mechanisms.CodeContext
mechanism_attribution_BPTrace = gap_mechanisms.BPTrace
mechanism_attribution_contexts = {}
mechanism_attribution_gf2_rank = gap_mechanisms.gf2_rank
mechanism_attribution_graph_adjacency = gap_mechanisms.graph_adjacency

def mechanism_attribution_context_for(level, p):
    return gap_mechanisms.context_for(
        level,
        p,
        CA_9_2_5=CA_9_2_5,
        CONTEXTS=mechanism_attribution_contexts,
        CircuitBuilder=CircuitBuilder,
        CodeContext=mechanism_attribution_CodeContext,
        ErrorRate=ErrorRate,
        TriangularCode=TriangularCode,
        detector_error_model_to_check_matrices=detector_error_model_to_check_matrices,
        gf2_rank=mechanism_attribution_gf2_rank,
        graph_adjacency=mechanism_attribution_graph_adjacency,
    )

def mechanism_attribution_transformed_context(context, transform):
    return gap_mechanisms.transformed_context(context, transform, graph_adjacency=mechanism_attribution_graph_adjacency)

def mechanism_attribution_trace_bp(
    context,
    syndrome,
    iterations,
    update_rule='min_sum',
    scaling=1.0,
    offset=0.0,
    damping=0.0,
    schedule='parallel',
    rng=None,
    prior_llr=None,
    initial_message_sigma=0.0,
):
    return gap_mechanisms.trace_bp(context, syndrome, iterations, update_rule, scaling, offset, damping, schedule, rng, prior_llr, initial_message_sigma, BPTrace=mechanism_attribution_BPTrace)

def mechanism_attribution_logical_label(context, correction):
    return gap_mechanisms.logical_label(context, correction, bits=mechanism_attribution_bits)

def mechanism_attribution_outcome(
    context,
    correction,
    syndrome,
    truth,
    error=None,
):
    return gap_mechanisms.outcome(context, correction, syndrome, truth, error, bits=mechanism_attribution_bits)

mechanism_attribution_prior_cost = gap_mechanisms.prior_cost

def mechanism_attribution_full_osd_order(context):
    return gap_mechanisms.full_osd_order(context, CONFIG=mechanism_attribution_config)

def mechanism_attribution_osd_decode(
    context,
    syndrome,
    llr,
    order,
):
    return gap_mechanisms.osd_decode(context, syndrome, llr, order, ordered_statistics_postprocess=ordered_statistics_postprocess)

def mechanism_attribution_standard_bposd(context, syndrome, trace):
    return gap_mechanisms.standard_bposd(context, syndrome, trace, CONFIG=mechanism_attribution_config, full_osd_order=mechanism_attribution_full_osd_order, osd_decode=mechanism_attribution_osd_decode)

mechanism_attribution_trace_metrics = gap_mechanisms.trace_metrics

def mechanism_attribution_tesseract_predictions(context, syndromes):
    return gap_mechanisms.tesseract_predictions(context, syndromes, CONFIG=mechanism_attribution_config, Tesseract=Tesseract)

def mechanism_attribution_exact_sector_costs(context, syndrome):
    return gap_mechanisms.exact_sector_costs(context, syndrome, CONFIG=mechanism_attribution_config, bits=mechanism_attribution_bits)


## Paired physical-shot population and gap cohort

Generate the paired population and identify standard-failure/Tesseract-success cases. Keep this enriched diagnostic cohort separate from the unbiased population used for final LER.


In [ ]:
mechanism_attribution_population_path = mechanism_attribution_data_paths.output('paired_population_csv')
TRAJECTORY_PATH = mechanism_attribution_data_paths.output('gap_bp_trajectories_csv')

def material(row):
    context = mechanism_attribution_context_for(int(row.level), float(row.p))
    syndrome = mechanism_attribution_bit_array(row.syndrome, context.H.shape[0])
    truth = mechanism_attribution_bit_array(row.truth, context.L.shape[0])
    error = mechanism_attribution_bit_array(row.error, context.H.shape[1])

    return (context, syndrome, truth, error)

def decode_population_shot(
    context,
    syndrome,
    truth,
    error,
    tess,
    tess_runtime,
    tess_compile_amortized,
    shot_id,
    population_role,
):
    trace = mechanism_attribution_trace_bp(context, syndrome, mechanism_attribution_config['continued_iterations'])
    correction, endpoint, osd_used, runtime = mechanism_attribution_standard_bposd(context, syndrome, trace)
    valid, correct, exact, sector = mechanism_attribution_outcome(context, correction, syndrome, truth, error)
    bp_valid = np.flatnonzero(trace.valid[1:mechanism_attribution_config['bp_standard_iterations'] + 1]) + 1
    bp_endpoint = int(bp_valid[0]) if len(bp_valid) else mechanism_attribution_config['bp_standard_iterations']
    bp_outcome = mechanism_attribution_outcome(context, trace.hard[bp_endpoint], syndrome, truth, error)
    metrics = mechanism_attribution_trace_metrics(context, trace, endpoint)

    return {
        'shot_id': shot_id,
        'level': context.level,
        'distance': context.distance,
        'p': context.p,
        'split': mechanism_attribution_split_for(shot_id),
        'population_role': population_role,
        'syndrome': mechanism_attribution_bits(syndrome),
        'truth': mechanism_attribution_bits(truth),
        'error': mechanism_attribution_bits(error),
        'error_weight': int(error.sum()),
        'standard_valid': valid,
        'standard_correct': correct,
        'standard_exact': exact,
        'standard_sector': sector,
        'standard_endpoint': endpoint,
        'standard_osd_used': osd_used,
        'standard_runtime': runtime,
        'standard_bp_endpoint': bp_endpoint,
        'standard_bp_valid': bp_outcome[0],
        'standard_bp_correct': bp_outcome[1],
        'standard_bp_sector': bp_outcome[3],
        'tesseract_correct': int(np.array_equal(tess, truth)),
        'tesseract_sector': mechanism_attribution_bits(tess),
        'tesseract_runtime': tess_runtime,
        'tesseract_compile_amortized': tess_compile_amortized,
        'first_valid': -1 if trace.first_valid is None else trace.first_valid,
        **metrics,
    }

if mechanism_attribution_stage_cached([mechanism_attribution_population_path]):
    mechanism_attribution_population = mechanism_attribution_read_csv(mechanism_attribution_population_path)
else:
    mechanism_attribution_rows = []
    point_specs = [(level, p, 'primary') for level in mechanism_attribution_config['primary_levels'] for p in mechanism_attribution_config['p_values']]
    point_specs += [(level, p, 'validation') for level in mechanism_attribution_config['validation_levels'] for p in mechanism_attribution_config['validation_p_values']]

    for mechanism_attribution_point_index, (mechanism_attribution_level, mechanism_attribution_p, mechanism_attribution_point_role) in enumerate(point_specs):
        mechanism_attribution_context = mechanism_attribution_context_for(mechanism_attribution_level, mechanism_attribution_p)
        mechanism_attribution_point_rows = []
        mechanism_attribution_seen = 0

        while mechanism_attribution_seen < mechanism_attribution_config['maximum_shots_per_point']:
            mechanism_attribution_amount = min(mechanism_attribution_config['sampling_batch_size'], mechanism_attribution_config['maximum_shots_per_point'] - mechanism_attribution_seen)
            mechanism_attribution_sampler = mechanism_attribution_context.dem.compile_sampler(seed=mechanism_attribution_config['random_seed'] + mechanism_attribution_point_index * 100003 + mechanism_attribution_seen)
            mechanism_attribution_syndromes, mechanism_attribution_truths, mechanism_attribution_errors = mechanism_attribution_sampler.sample(shots=mechanism_attribution_amount, return_errors=True)
            mechanism_attribution_tess, tess_runtimes, tess_compile_amortized = mechanism_attribution_tesseract_predictions(mechanism_attribution_context, mechanism_attribution_syndromes)
            mechanism_attribution_iterator = zip(mechanism_attribution_syndromes, mechanism_attribution_truths, mechanism_attribution_errors, mechanism_attribution_tess, tess_runtimes)

            for mechanism_attribution_local, mechanism_attribution_values in enumerate(tqdm(mechanism_attribution_iterator, total=mechanism_attribution_amount, desc=f'population d={mechanism_attribution_context.distance} p={mechanism_attribution_p:g}', disable=mechanism_attribution_config['print_level'] == 1)):
                mechanism_attribution_syndrome, mechanism_attribution_truth, mechanism_attribution_error, mechanism_attribution_prediction = [np.asarray(value, dtype=np.uint8) for value in mechanism_attribution_values[:4]]
                mechanism_attribution_tess_runtime = float(mechanism_attribution_values[4])
                mechanism_attribution_shot_id = f'L{mechanism_attribution_level}-p{mechanism_attribution_point_index}-b{mechanism_attribution_seen}-i{mechanism_attribution_local}'
                mechanism_attribution_population_role = 'unbiased' if mechanism_attribution_seen + mechanism_attribution_local < mechanism_attribution_config['minimum_shots_per_point'] else 'gap_enrichment'
                mechanism_attribution_point_rows.append(decode_population_shot(mechanism_attribution_context, mechanism_attribution_syndrome, mechanism_attribution_truth, mechanism_attribution_error, mechanism_attribution_prediction, mechanism_attribution_tess_runtime, tess_compile_amortized, mechanism_attribution_shot_id, mechanism_attribution_population_role))

            mechanism_attribution_seen += mechanism_attribution_amount
            mechanism_attribution_frame = pd.DataFrame(mechanism_attribution_point_rows)
            gap_count = int(((mechanism_attribution_frame.standard_correct == 0) & (mechanism_attribution_frame.tesseract_correct == 1)).sum())
            mechanism_attribution_log(2, {
                'stage': 'population',
                'distance': mechanism_attribution_context.distance,
                'p': mechanism_attribution_p,
                'shots': mechanism_attribution_seen,
                'gap': gap_count,
            })
            mechanism_attribution_enough_gap = gap_count >= mechanism_attribution_config['target_gap_cases_per_point'] if mechanism_attribution_point_role == 'primary' else True

            if mechanism_attribution_seen >= mechanism_attribution_config['minimum_shots_per_point'] and mechanism_attribution_enough_gap:
                break

        mechanism_attribution_rows.extend(mechanism_attribution_point_rows)

    mechanism_attribution_population = mechanism_attribution_save_csv(pd.DataFrame(mechanism_attribution_rows), mechanism_attribution_population_path)

mechanism_attribution_population['gap'] = ((mechanism_attribution_population.standard_correct == 0) & (mechanism_attribution_population.tesseract_correct == 1)).astype(int)
unbiased_population = mechanism_attribution_population[mechanism_attribution_population.population_role == 'unbiased'].copy()
assert unbiased_population.groupby(['distance', 'p']).size().eq(mechanism_attribution_config['minimum_shots_per_point']).all()
mechanism_attribution_primary_distances = {mechanism_attribution_context_for(level, mechanism_attribution_config['p_values'][0]).distance for level in mechanism_attribution_config['primary_levels']}
mechanism_attribution_gap_cases = mechanism_attribution_population[(mechanism_attribution_population.gap == 1) & mechanism_attribution_population.distance.isin(mechanism_attribution_primary_distances)].copy()
mechanism_attribution_gap_cases = mechanism_attribution_gap_cases.groupby(['distance', 'p'], sort=False).head(mechanism_attribution_config['analysis_gap_limit_per_point']).reset_index(drop=True)
assert len(mechanism_attribution_gap_cases), 'No BPOSD-wrong/Tesseract-correct gap cases were collected'
population_summary = unbiased_population.groupby(['distance', 'p'], as_index=False).agg(
    shots=('shot_id', 'size'),
    standard_LER=('standard_correct', lambda x: 1 - x.mean()),
    tesseract_LER=('tesseract_correct', lambda x: 1 - x.mean()),
    gap_rate=('gap', 'mean'),
    gap_cases=('gap', 'sum'),
)
mechanism_attribution_save_csv(population_summary, mechanism_attribution_data_paths.output('population_summary_csv'))
population_summary


## Experiments 1–2: strict early stop and OSD-order decomposition

Continue beyond early acceptance and vary OSD order on the same inputs. These interventions distinguish a stopping opportunity from sensitivity to search depth.


In [ ]:
EARLY_PATH = mechanism_attribution_data_paths.output('experiment_1_early_stop_csv')
OSD_PATH = mechanism_attribution_data_paths.output('experiment_2_osd_orders_csv')
TRAJECTORY_PATH = mechanism_attribution_data_paths.output('gap_bp_trajectories_csv')

if mechanism_attribution_stage_cached([EARLY_PATH, OSD_PATH, TRAJECTORY_PATH]):
    early_stop = mechanism_attribution_read_csv(EARLY_PATH)
    osd_orders = mechanism_attribution_read_csv(OSD_PATH)
    gap_trajectories = mechanism_attribution_read_csv(TRAJECTORY_PATH)
else:
    early_rows, osd_rows, trajectory_rows = ([], [], [])
    for mechanism_attribution_case in tqdm(mechanism_attribution_gap_cases.itertuples(index=False), total=len(mechanism_attribution_gap_cases), desc='Experiments 1-2', disable=mechanism_attribution_config['print_level'] == 1):
        mechanism_attribution_context, mechanism_attribution_syndrome, mechanism_attribution_truth, mechanism_attribution_error = material(mechanism_attribution_case)
        mechanism_attribution_trace = mechanism_attribution_trace_bp(mechanism_attribution_context, mechanism_attribution_syndrome, mechanism_attribution_config['continued_iterations'])
        standard_correction, mechanism_attribution_endpoint, mechanism_attribution_, mechanism_attribution_ = mechanism_attribution_standard_bposd(mechanism_attribution_context, mechanism_attribution_syndrome, mechanism_attribution_trace)
        valid_iterations = np.flatnonzero(mechanism_attribution_trace.valid)
        correct_iterations = [int(t) for t in valid_iterations if mechanism_attribution_outcome(mechanism_attribution_context, mechanism_attribution_trace.hard[t], mechanism_attribution_syndrome, mechanism_attribution_truth)[1]]
        later = [t for t in correct_iterations if t > mechanism_attribution_endpoint]
        valid_sectors = [mechanism_attribution_logical_label(mechanism_attribution_context, mechanism_attribution_trace.hard[t]) for t in valid_iterations]
        persistent = bool(later and all((mechanism_attribution_trace.valid[t] and mechanism_attribution_outcome(mechanism_attribution_context, mechanism_attribution_trace.hard[t], mechanism_attribution_syndrome, mechanism_attribution_truth)[1] for t in range(later[0], len(mechanism_attribution_trace.valid)))))
        early_rows.append({
            'shot_id': mechanism_attribution_case.shot_id,
            'distance': mechanism_attribution_case.distance,
            'p': mechanism_attribution_case.p,
            'standard_endpoint': mechanism_attribution_endpoint,
            'later_BP_correct': int(bool(later)),
            'delay_to_later_correct': np.nan if not later else later[0] - mechanism_attribution_endpoint,
            'valid_corrections': len(valid_iterations),
            'logical_sector_switches': int(np.sum(np.asarray(valid_sectors[1:]) != np.asarray(valid_sectors[:-1]))) if len(valid_sectors) > 1 else 0,
            'later_correct_persistent': int(persistent),
            'later_correct_transient': int(bool(later) and (not persistent)),
        })
        endpoint_llr = mechanism_attribution_trace.llr[mechanism_attribution_endpoint]
        order_values = mechanism_attribution_config['osd_orders'] + [mechanism_attribution_full_osd_order(mechanism_attribution_context)]

        for label, mechanism_attribution_order in [(str(value), value) for value in mechanism_attribution_config['osd_orders']] + [('configured_full', mechanism_attribution_full_osd_order(mechanism_attribution_context))]:
            mechanism_attribution_correction, mechanism_attribution_runtime = mechanism_attribution_osd_decode(mechanism_attribution_context, mechanism_attribution_syndrome, endpoint_llr, mechanism_attribution_order)
            mechanism_attribution_valid, mechanism_attribution_correct, mechanism_attribution_exact, mechanism_attribution_sector = mechanism_attribution_outcome(mechanism_attribution_context, mechanism_attribution_correction, mechanism_attribution_syndrome, mechanism_attribution_truth, mechanism_attribution_error)
            osd_rows.append({
                'shot_id': mechanism_attribution_case.shot_id,
                'distance': mechanism_attribution_case.distance,
                'p': mechanism_attribution_case.p,
                'osd_label': label,
                'osd_order': mechanism_attribution_order,
                'candidate_count': osd_candidate_count(mechanism_attribution_context.H.shape[1] - mechanism_attribution_context.H.shape[0], mechanism_attribution_order),
                'syndrome_consistent': mechanism_attribution_valid,
                'logical_correct': mechanism_attribution_correct,
                'exact_error': mechanism_attribution_exact,
                'logical_sector': mechanism_attribution_sector,
                'runtime': mechanism_attribution_runtime,
            })

        for iteration in range(len(mechanism_attribution_trace.valid)):
            trajectory_rows.append({
                'shot_id': mechanism_attribution_case.shot_id,
                'iteration': iteration,
                'syndrome_consistent': int(mechanism_attribution_trace.valid[iteration]),
                'logical_sector': mechanism_attribution_logical_label(mechanism_attribution_context, mechanism_attribution_trace.hard[iteration]),
                'logical_correct': mechanism_attribution_outcome(mechanism_attribution_context, mechanism_attribution_trace.hard[iteration], mechanism_attribution_syndrome, mechanism_attribution_truth)[1],
                'hard': mechanism_attribution_bits(mechanism_attribution_trace.hard[iteration]),
                'llr_json': json.dumps(mechanism_attribution_trace.llr[iteration].tolist()),
                'c2v_json': json.dumps(mechanism_attribution_trace.c2v[iteration].tolist()),
                'v2c_json': json.dumps(mechanism_attribution_trace.v2c[iteration].tolist()),
            })

    early_stop = mechanism_attribution_save_csv(pd.DataFrame(early_rows), EARLY_PATH)
    osd_orders = mechanism_attribution_save_csv(pd.DataFrame(osd_rows), OSD_PATH)
    gap_trajectories = mechanism_attribution_save_csv(pd.DataFrame(trajectory_rows), TRAJECTORY_PATH)

q_min_rows = []
for mechanism_attribution_shot_id, mechanism_attribution_group in osd_orders.groupby('shot_id'):
    mechanism_attribution_successful = mechanism_attribution_group[mechanism_attribution_group.logical_correct == 1].sort_values('osd_order')
    q_min_rows.append({
        'shot_id': mechanism_attribution_shot_id,
        'q_min': 'unrepaired' if mechanism_attribution_successful.empty else mechanism_attribution_successful.iloc[0].osd_label,
    })

q_min = pd.DataFrame(q_min_rows).merge(early_stop[['shot_id', 'later_BP_correct']], on='shot_id')
mechanism_attribution_save_csv(q_min.groupby(['later_BP_correct', 'q_min'], as_index=False).size().rename(columns={'size': 'cases'}), mechanism_attribution_data_paths.output('osd_order_decomposition_csv'))
mechanism_attribution_save_csv(
    early_stop.groupby(['distance', 'p'], as_index=False).agg(
        gap_cases=('shot_id', 'size'),
        early_stop_repair_rate=('later_BP_correct', 'mean'),
        mean_delay=('delay_to_later_correct', 'mean'),
        mean_sector_switches=('logical_sector_switches', 'mean'),
    ),
    mechanism_attribution_data_paths.output('early_stop_summary_csv'),
)
q_min.head()


## Experiments 3–4: equivalent PCM ensemble and paired PCM mechanisms

Decode equivalent PCM views and compare successful with failing representations. The constraint problem must be preserved by transforming the syndrome consistently with the checks.


In [ ]:
@dataclass(frozen=True)
class PCMView:
    view_id: int
    transform: np.ndarray
    kind: str

def redundant_row_candidates(H, max_weight):
    rows, seen = ([], set())
    for left, right in combinations(range(H.shape[0]), 2):
        row = H[left] ^ H[right]
        key = row.tobytes()
        if 0 < int(row.sum()) <= max_weight and key not in seen:
            seen.add(key)
            rows.append((left, right, row))

    assert rows, 'No redundant-row candidates'
    return rows

def random_pcm_views(context, count, rng):
    m = context.H.shape[0]
    candidates = redundant_row_candidates(context.H, mechanism_attribution_config['candidate_max_row_weight'])
    views = [PCMView(0, np.eye(m, dtype=np.uint8), 'standard')]

    for view_id in range(1, count + 1):
        if view_id % 2:
            transform = np.eye(m, dtype=np.uint8)
            operations = int(rng.integers(1, max(2, m // 2)))
            for _ in range(operations):
                left, right = rng.choice(m, 2, replace=False)
                transform[left] ^= transform[right]

            assert mechanism_attribution_gf2_rank(transform) == m
            kind = 'basis_replacement'
        else:
            amount = int(rng.choice(mechanism_attribution_config['pcm_rows_per_view']))
            selected = rng.choice(len(candidates), min(amount, len(candidates)), replace=False)
            extra = []

            for index in np.atleast_1d(selected):
                left, right, _ = candidates[int(index)]
                row = np.zeros(m, dtype=np.uint8)
                row[[left, right]] = 1
                extra.append(row)

            transform = np.vstack([np.eye(m, dtype=np.uint8), np.asarray(extra)])
            kind = 'redundant_addition'

        views.append(PCMView(view_id, transform, kind))

    return views

def tanner_graph(H):
    graph = nx.Graph()
    graph.add_nodes_from([f'c{i}' for i in range(H.shape[0])], bipartite=0)
    graph.add_nodes_from([f'v{i}' for i in range(H.shape[1])], bipartite=1)
    checks, variables = np.nonzero(H)
    graph.add_edges_from(((f'c{c}', f'v{v}') for c, v in zip(checks, variables)))

    return graph

def cycle_counts(H):
    shared = H @ H.T
    four = int(sum((comb(int(value), 2) for value in shared[np.triu_indices(H.shape[0], 1)] if value >= 2)))
    six = 0
    supports = [set(np.flatnonzero(H[check])) for check in range(H.shape[0])]

    for first, second, third in combinations(range(H.shape[0]), 3):
        for v12 in supports[first] & supports[second]:
            for v23 in supports[second] & supports[third]:
                if v23 == v12:
                    continue
                six += sum((v31 not in {v12, v23} for v31 in supports[third] & supports[first]))

    graph = tanner_graph(H)
    girth_value = nx.girth(graph)
    girth = np.nan if not np.isfinite(girth_value) else int(girth_value)
    cycle_rank = graph.number_of_edges() - graph.number_of_nodes() + nx.number_connected_components(graph)

    return (four, int(six), girth, int(cycle_rank))

def graph_metrics(H, low_variables=None, repair_variables=None):
    four, six, girth, cycle_rank = cycle_counts(H)
    row_weights = H.sum(axis=1)
    variable_degrees = H.sum(axis=0)
    low_variables = np.asarray([], dtype=int) if low_variables is None else np.asarray(low_variables, dtype=int)
    repair_variables = np.asarray([], dtype=int) if repair_variables is None else np.asarray(repair_variables, dtype=int)

    return {
        'rows': H.shape[0],
        'edges': int(H.sum()),
        'mean_row_weight': float(row_weights.mean()),
        'max_row_weight': int(row_weights.max()),
        'mean_variable_degree': float(variable_degrees.mean()),
        'four_cycles': four,
        'six_cycles': six,
        'girth': girth,
        'cycle_rank': cycle_rank,
        'checks_touching_low_reliability': int(np.any(H[:, low_variables], axis=1).sum()) if len(low_variables) else 0,
        'checks_touching_repair_support': int(np.any(H[:, repair_variables], axis=1).sum()) if len(repair_variables) else 0,
    }

PCM_PATH = mechanism_attribution_trace_dir / 'experiment_3_pcm_views.csv'
PCM_PAIR_PATH = mechanism_attribution_trace_dir / 'experiment_4_pcm_pairs.csv'
if mechanism_attribution_stage_cached([PCM_PATH, PCM_PAIR_PATH]):
    pcm_views = mechanism_attribution_read_csv(PCM_PATH)
    pcm_pairs = mechanism_attribution_read_csv(PCM_PAIR_PATH)
else:
    pcm_rows, mechanism_attribution_pair_rows = ([], [])
    selected_ids = set(mechanism_attribution_gap_cases.head(mechanism_attribution_config['selected_pcm_case_limit']).shot_id)
    for mechanism_attribution_case_index, mechanism_attribution_case in enumerate(tqdm(mechanism_attribution_gap_cases.itertuples(index=False), total=len(mechanism_attribution_gap_cases), desc='Experiments 3-4', disable=mechanism_attribution_config['print_level'] == 1)):
        mechanism_attribution_context, mechanism_attribution_syndrome, mechanism_attribution_truth, mechanism_attribution_error = material(mechanism_attribution_case)
        count = mechanism_attribution_config['selected_pcm_views'] if mechanism_attribution_case.shot_id in selected_ids else mechanism_attribution_config['pcm_views']
        mechanism_attribution_rng = np.random.default_rng(mechanism_attribution_config['random_seed'] + mechanism_attribution_case_index)
        mechanism_attribution_views = random_pcm_views(mechanism_attribution_context, count, mechanism_attribution_rng)
        case_records = []
        baseline_hard = None

        for view in mechanism_attribution_views:
            mechanism_attribution_view_context = mechanism_attribution_transformed_context(mechanism_attribution_context, view.transform)
            mechanism_attribution_view_syndrome = view.transform @ mechanism_attribution_syndrome % 2
            mechanism_attribution_trace = mechanism_attribution_trace_bp(mechanism_attribution_view_context, mechanism_attribution_view_syndrome, mechanism_attribution_config['bp_standard_iterations'])
            mechanism_attribution_endpoint = mechanism_attribution_trace.first_valid if mechanism_attribution_trace.first_valid is not None else mechanism_attribution_config['bp_standard_iterations']
            mechanism_attribution_correction = mechanism_attribution_trace.hard[mechanism_attribution_endpoint]
            mechanism_attribution_valid, mechanism_attribution_correct, mechanism_attribution_exact, mechanism_attribution_sector = mechanism_attribution_outcome(mechanism_attribution_context, mechanism_attribution_correction, mechanism_attribution_syndrome, mechanism_attribution_truth, mechanism_attribution_error)
            mechanism_attribution_metrics = mechanism_attribution_trace_metrics(mechanism_attribution_view_context, mechanism_attribution_trace, mechanism_attribution_endpoint)

            mechanism_attribution_record = {
                'shot_id': mechanism_attribution_case.shot_id,
                'view_id': view.view_id,
                'view_kind': view.kind,
                'transform_json': json.dumps(view.transform.tolist()),
                'syndrome_consistent': mechanism_attribution_valid,
                'logical_correct': mechanism_attribution_correct,
                'logical_sector': mechanism_attribution_sector,
                'converged': int(mechanism_attribution_trace.first_valid is not None),
                'iterations': mechanism_attribution_endpoint,
                'prior_cost': mechanism_attribution_prior_cost(mechanism_attribution_context, mechanism_attribution_correction),
                'posterior_llr_json': json.dumps(mechanism_attribution_trace.llr[mechanism_attribution_endpoint].tolist()),
                **mechanism_attribution_metrics,
            }

            pcm_rows.append(mechanism_attribution_record)
            case_records.append((mechanism_attribution_record, view, mechanism_attribution_view_context, mechanism_attribution_trace, mechanism_attribution_correction))
            if view.view_id == 0:
                baseline_hard = mechanism_attribution_correction.copy()

        mechanism_attribution_failing = next((item for item in case_records if item[0]['logical_correct'] == 0))
        successful_items = [item for item in case_records if item[0]['logical_correct'] == 1]
        for mechanism_attribution_successful in successful_items:
            mechanism_attribution_pair_id = f'{mechanism_attribution_case.shot_id}:v{mechanism_attribution_successful[1].view_id}'
            mechanism_attribution_repair_support = np.flatnonzero(mechanism_attribution_failing[4] ^ mechanism_attribution_successful[4])
            mechanism_attribution_low = np.argsort(np.abs(mechanism_attribution_failing[3].llr[mechanism_attribution_failing[0]['iterations']]))[:mechanism_attribution_config['ambiguity_top_k']]

            for mechanism_attribution_role, item in [('failing', mechanism_attribution_failing), ('successful', mechanism_attribution_successful)]:
                mechanism_attribution_record, view, mechanism_attribution_view_context, mechanism_attribution_trace, mechanism_attribution_correction = item
                mechanism_attribution_pair_rows.append({
                    'pair_id': mechanism_attribution_pair_id,
                    'shot_id': mechanism_attribution_case.shot_id,
                    'role': mechanism_attribution_role,
                    'view_id': view.view_id,
                    **mechanism_attribution_trace_metrics(mechanism_attribution_view_context, mechanism_attribution_trace, mechanism_attribution_record['iterations']),
                    **graph_metrics(mechanism_attribution_view_context.H, mechanism_attribution_low, mechanism_attribution_repair_support),
                })

    pcm_views = mechanism_attribution_save_csv(pd.DataFrame(pcm_rows), PCM_PATH)
    mechanism_attribution_pair_columns = ['pair_id', 'shot_id', 'role', 'view_id', 'message_instability', 'message_instability_max', 'belief_instability', 'sign_flip', 'extrinsic_conflict', 'posterior_abs_mean', 'reliability_rank', 'rows', 'edges', 'mean_row_weight', 'max_row_weight', 'mean_variable_degree', 'four_cycles', 'six_cycles', 'girth', 'cycle_rank', 'checks_touching_low_reliability', 'checks_touching_repair_support']
    pcm_pairs = mechanism_attribution_save_csv(pd.DataFrame(mechanism_attribution_pair_rows, columns=mechanism_attribution_pair_columns), PCM_PAIR_PATH)

pcm_alternative_views = pcm_views[pcm_views.view_id != 0]
pcm_summary = pcm_alternative_views.groupby('shot_id', as_index=False).agg(PCM_any_correct=('logical_correct', 'max'), PCM_success_fraction=('logical_correct', 'mean'), PCM_unique_sectors=('logical_sector', 'nunique'))
mechanism_attribution_save_csv(pcm_summary, mechanism_attribution_trace_dir / 'experiment_3_pcm_case_summary.csv')
pcm_case_summary = mechanism_attribution_gap_cases[['shot_id', 'distance', 'p']].merge(pcm_summary, on='shot_id')
mechanism_attribution_save_csv(
    pcm_case_summary.groupby(['distance', 'p'], as_index=False).agg(
        gap_cases=('shot_id', 'size'),
        PCM_any_correct_rate=('PCM_any_correct', 'mean'),
        mean_PCM_success_fraction=('PCM_success_fraction', 'mean'),
        mean_unique_sectors=('PCM_unique_sectors', 'mean'),
    ),
    mechanism_attribution_data_paths.output('pcm_view_success_summary_csv'),
)

if len(pcm_pairs):
    paired_wide = pcm_pairs.pivot(index=['pair_id', 'shot_id'], columns='role')
    paired_differences = pd.DataFrame({column: paired_wide[column]['successful'] - paired_wide[column]['failing'] for column in ['message_instability', 'message_instability_max', 'belief_instability', 'sign_flip', 'extrinsic_conflict', 'four_cycles', 'six_cycles', 'cycle_rank', 'mean_row_weight']}).reset_index()
else:
    paired_differences = pd.DataFrame(columns=['pair_id', 'shot_id', 'message_instability', 'belief_instability', 'sign_flip', 'extrinsic_conflict', 'four_cycles', 'six_cycles', 'cycle_rank', 'mean_row_weight'])

mechanism_attribution_save_csv(paired_differences, mechanism_attribution_trace_dir / 'experiment_4_paired_pcm_differences.csv')
if len(paired_differences):
    paired_pcm_summary = paired_differences.drop(columns=['pair_id', 'shot_id']).agg(['count', 'mean', 'median']).T.reset_index(names='metric')
else:
    paired_pcm_summary = pd.DataFrame(columns=['metric', 'count', 'mean', 'median'])
mechanism_attribution_save_csv(paired_pcm_summary, mechanism_attribution_data_paths.output('paired_pcm_summary_csv'))
pcm_summary.head()


## Experiment 5: multi-basin random restarts

Repeat the configured restart experiments to test whether alternative trajectories reach different endpoints or logical sectors. A restart repair indicates basin sensitivity, not a unique explanation for the original failure.


In [ ]:
RESTART_PATH = mechanism_attribution_trace_dir / 'experiment_5_restarts.csv'
if mechanism_attribution_stage_cached([RESTART_PATH]):
    mechanism_attribution_restart_trials = mechanism_attribution_read_csv(RESTART_PATH)
else:
    mechanism_attribution_rows = []
    for mechanism_attribution_case_index, mechanism_attribution_case in enumerate(tqdm(mechanism_attribution_gap_cases.itertuples(index=False), total=len(mechanism_attribution_gap_cases), desc='Experiment 5', disable=mechanism_attribution_config['print_level'] == 1)):
        mechanism_attribution_context, mechanism_attribution_syndrome, mechanism_attribution_truth, mechanism_attribution_error = material(mechanism_attribution_case)
        mechanism_attribution_successful = pcm_views[(pcm_views.shot_id == mechanism_attribution_case.shot_id) & (pcm_views.logical_correct == 1)]
        mechanism_attribution_transforms = [('standard_PCM', np.eye(mechanism_attribution_context.H.shape[0], dtype=np.uint8))]

        if len(mechanism_attribution_successful):
            mechanism_attribution_transforms.append(('successful_PCM', np.asarray(json.loads(mechanism_attribution_successful.iloc[0].transform_json), dtype=np.uint8)))
        for pcm_name, mechanism_attribution_transform in mechanism_attribution_transforms:
            mechanism_attribution_view_context = mechanism_attribution_transformed_context(mechanism_attribution_context, mechanism_attribution_transform)
            mechanism_attribution_view_syndrome = mechanism_attribution_transform @ mechanism_attribution_syndrome % 2
            for mechanism_attribution_restart in range(mechanism_attribution_config['restart_count']):
                mechanism_attribution_rng = np.random.default_rng(mechanism_attribution_config['random_seed'] + mechanism_attribution_case_index * 10000 + mechanism_attribution_restart)
                mechanism_attribution_kind = ['message_perturbation', 'prior_perturbation', 'random_serial_schedule', 'random_update_order'][mechanism_attribution_restart % 4]
                prior = mechanism_attribution_context.prior_llr.copy()
                sigma = 0.0
                mechanism_attribution_schedule = 'parallel'

                if mechanism_attribution_kind == 'message_perturbation':
                    sigma = mechanism_attribution_config['restart_message_sigma']
                elif mechanism_attribution_kind == 'prior_perturbation':
                    prior *= 1 + mechanism_attribution_rng.normal(0, mechanism_attribution_config['restart_prior_sigma'], len(prior))
                elif mechanism_attribution_kind == 'random_serial_schedule':
                    mechanism_attribution_schedule = 'random_serial_fixed'
                else:
                    mechanism_attribution_schedule = 'random_serial'

                mechanism_attribution_trace = mechanism_attribution_trace_bp(
                    mechanism_attribution_view_context,
                    mechanism_attribution_view_syndrome,
                    mechanism_attribution_config['bp_standard_iterations'],
                    rng=mechanism_attribution_rng,
                    prior_llr=prior,
                    initial_message_sigma=sigma,
                    schedule=mechanism_attribution_schedule,
                )
                mechanism_attribution_endpoint = mechanism_attribution_trace.first_valid if mechanism_attribution_trace.first_valid is not None else mechanism_attribution_config['bp_standard_iterations']
                mechanism_attribution_correction = mechanism_attribution_trace.hard[mechanism_attribution_endpoint]
                mechanism_attribution_valid, mechanism_attribution_correct, mechanism_attribution_, mechanism_attribution_sector = mechanism_attribution_outcome(mechanism_attribution_context, mechanism_attribution_correction, mechanism_attribution_syndrome, mechanism_attribution_truth)
                mechanism_attribution_rows.append({
                    'shot_id': mechanism_attribution_case.shot_id,
                    'pcm': pcm_name,
                    'restart': mechanism_attribution_restart,
                    'restart_kind': mechanism_attribution_kind,
                    'syndrome_consistent': mechanism_attribution_valid,
                    'logical_correct': mechanism_attribution_correct,
                    'logical_sector': mechanism_attribution_sector,
                })

    mechanism_attribution_restart_trials = mechanism_attribution_save_csv(pd.DataFrame(mechanism_attribution_rows), RESTART_PATH)

restart_summary = mechanism_attribution_restart_trials.groupby(['shot_id', 'pcm'], as_index=False).agg(restart_correct_fraction=('logical_correct', 'mean'), restart_unique_sectors=('logical_sector', 'nunique'), restart_valid_fraction=('syndrome_consistent', 'mean'))
mechanism_attribution_save_csv(restart_summary, mechanism_attribution_trace_dir / 'experiment_5_restart_case_summary.csv')
mechanism_attribution_save_csv(
    restart_summary.groupby('pcm', as_index=False).agg(
        cases=('shot_id', 'size'),
        mean_correct_fraction=('restart_correct_fraction', 'mean'),
        multiple_sector_rate=('restart_unique_sectors', lambda x: np.mean(np.asarray(x) > 1)),
        mean_valid_fraction=('restart_valid_fraction', 'mean'),
    ),
    mechanism_attribution_data_paths.output('restart_basin_summary_csv'),
)
restart_summary.head()


## Experiments 6–7: update rules, damping and scaling

Compare update rules, damping and scaling while retaining the paired decoding input. This isolates numerical-dynamics sensitivity from the graph-view interventions above.


In [ ]:
UPDATE_PATH = mechanism_attribution_trace_dir / 'experiment_6_update_rules.csv'
GRID_PATH = mechanism_attribution_trace_dir / 'experiment_7_damping_scaling.csv'
if mechanism_attribution_stage_cached([UPDATE_PATH, GRID_PATH]):
    update_trials = mechanism_attribution_read_csv(UPDATE_PATH)
    grid_trials = mechanism_attribution_read_csv(GRID_PATH)
else:
    update_rows, mechanism_attribution_grid_rows = ([], [])
    for mechanism_attribution_case_index, mechanism_attribution_case in enumerate(tqdm(mechanism_attribution_gap_cases.itertuples(index=False), total=len(mechanism_attribution_gap_cases), desc='Experiments 6-7', disable=mechanism_attribution_config['print_level'] == 1)):
        mechanism_attribution_context, mechanism_attribution_syndrome, mechanism_attribution_truth, mechanism_attribution_error = material(mechanism_attribution_case)
        variants = [('min_sum', 'min_sum', 1.0, 0.0)]
        variants += [(f'NMS_{alpha:g}', 'min_sum', alpha, 0.0) for alpha in mechanism_attribution_config['nms_alpha_values']]
        variants += [(f'OMS_{beta:g}', 'min_sum', 1.0, beta) for beta in mechanism_attribution_config['oms_beta_values']]
        variants += [('sum_product', 'sum_product', 1.0, 0.0)]

        for mechanism_attribution_name, mechanism_attribution_rule, mechanism_attribution_scaling, offset in variants:
            mechanism_attribution_trace = mechanism_attribution_trace_bp(mechanism_attribution_context, mechanism_attribution_syndrome, mechanism_attribution_config['bp_standard_iterations'], update_rule=mechanism_attribution_rule, scaling=mechanism_attribution_scaling, offset=offset)
            mechanism_attribution_endpoint = mechanism_attribution_trace.first_valid if mechanism_attribution_trace.first_valid is not None else mechanism_attribution_config['bp_standard_iterations']
            mechanism_attribution_valid, mechanism_attribution_correct, mechanism_attribution_, mechanism_attribution_sector = mechanism_attribution_outcome(mechanism_attribution_context, mechanism_attribution_trace.hard[mechanism_attribution_endpoint], mechanism_attribution_syndrome, mechanism_attribution_truth)
            update_rows.append({
                'shot_id': mechanism_attribution_case.shot_id,
                'variant': mechanism_attribution_name,
                'update_rule': mechanism_attribution_rule,
                'scaling': mechanism_attribution_scaling,
                'offset': offset,
                'syndrome_consistent': mechanism_attribution_valid,
                'logical_correct': mechanism_attribution_correct,
                'logical_sector': mechanism_attribution_sector,
                **mechanism_attribution_trace_metrics(mechanism_attribution_context, mechanism_attribution_trace, mechanism_attribution_endpoint),
            })

        for mechanism_attribution_scaling, mechanism_attribution_damping in product(mechanism_attribution_config['scaling_values'], mechanism_attribution_config['damping_values']):
            mechanism_attribution_trace = mechanism_attribution_trace_bp(mechanism_attribution_context, mechanism_attribution_syndrome, mechanism_attribution_config['bp_standard_iterations'], scaling=mechanism_attribution_scaling, damping=mechanism_attribution_damping)
            mechanism_attribution_endpoint = mechanism_attribution_trace.first_valid if mechanism_attribution_trace.first_valid is not None else mechanism_attribution_config['bp_standard_iterations']
            mechanism_attribution_valid, mechanism_attribution_correct, mechanism_attribution_, mechanism_attribution_sector = mechanism_attribution_outcome(mechanism_attribution_context, mechanism_attribution_trace.hard[mechanism_attribution_endpoint], mechanism_attribution_syndrome, mechanism_attribution_truth)
            mechanism_attribution_grid_rows.append({
                'shot_id': mechanism_attribution_case.shot_id,
                'scaling': mechanism_attribution_scaling,
                'damping': mechanism_attribution_damping,
                'syndrome_consistent': mechanism_attribution_valid,
                'logical_correct': mechanism_attribution_correct,
                'logical_sector': mechanism_attribution_sector,
                **mechanism_attribution_trace_metrics(mechanism_attribution_context, mechanism_attribution_trace, mechanism_attribution_endpoint),
            })

    update_trials = mechanism_attribution_save_csv(pd.DataFrame(update_rows), UPDATE_PATH)
    grid_trials = mechanism_attribution_save_csv(pd.DataFrame(mechanism_attribution_grid_rows), GRID_PATH)

mechanism_attribution_save_csv(
    update_trials.groupby('variant', as_index=False).agg(
        gap_cases=('shot_id', 'nunique'),
        repair_rate=('logical_correct', 'mean'),
        valid_rate=('syndrome_consistent', 'mean'),
        posterior_abs_mean=('posterior_abs_mean', 'mean'),
        message_instability=('message_instability', 'mean'),
    ),
    mechanism_attribution_data_paths.output('update_rule_summary_csv'),
)
mechanism_attribution_save_csv(
    grid_trials.groupby(['scaling', 'damping'], as_index=False).agg(
        gap_cases=('shot_id', 'nunique'),
        repair_rate=('logical_correct', 'mean'),
        valid_rate=('syndrome_consistent', 'mean'),
        message_instability=('message_instability', 'mean'),
    ),
    mechanism_attribution_data_paths.output('damping_scaling_summary_csv'),
)
grid_class_rows = []

for mechanism_attribution_shot_id, mechanism_attribution_group in grid_trials.groupby('shot_id'):
    scaling_repair = bool(mechanism_attribution_group[(mechanism_attribution_group.damping == 0) & (mechanism_attribution_group.scaling != 1)].logical_correct.max())
    damping_repair = bool(mechanism_attribution_group[(mechanism_attribution_group.scaling == 1) & (mechanism_attribution_group.damping > 0)].logical_correct.max())
    joint_repair = bool(mechanism_attribution_group[(mechanism_attribution_group.scaling != 1) & (mechanism_attribution_group.damping > 0)].logical_correct.max())

    if scaling_repair and damping_repair:
        repair_class = 'scaling_and_damping'
    elif scaling_repair:
        repair_class = 'scaling_only'
    elif damping_repair:
        repair_class = 'damping_only'
    elif joint_repair:
        repair_class = 'joint_only'
    else:
        repair_class = 'neither'

    grid_class_rows.append({
        'shot_id': mechanism_attribution_shot_id,
        'scaling_repaired': int(scaling_repair),
        'damping_repaired': int(damping_repair),
        'joint_repaired': int(joint_repair),
        'repair_class': repair_class,
    })

grid_repair_classes = mechanism_attribution_save_csv(pd.DataFrame(grid_class_rows), mechanism_attribution_trace_dir / 'experiment_7_repair_classes.csv')


## Experiments 8–9: exact logical-sector competition and prior sensitivity

Measure competition among logical sectors and sensitivity to priors. These are posterior diagnostics; they should not be substituted for an implementable runtime confidence signal.


In [ ]:
LOGICAL_PATH = mechanism_attribution_trace_dir / 'experiment_8_logical_costs.csv'
PRIOR_PATH = mechanism_attribution_trace_dir / 'experiment_9_prior_sensitivity.csv'
if mechanism_attribution_stage_cached([LOGICAL_PATH, PRIOR_PATH]):
    logical_costs = mechanism_attribution_read_csv(LOGICAL_PATH)
    prior_trials = mechanism_attribution_read_csv(PRIOR_PATH)
else:
    logical_rows, prior_rows = ([], [])
    logical_population = unbiased_population.copy()
    logical_population['cohort'] = np.select([(logical_population.standard_correct == 1) & (logical_population.tesseract_correct == 1), (logical_population.standard_correct == 0) & (logical_population.tesseract_correct == 1), (logical_population.standard_correct == 0) & (logical_population.tesseract_correct == 0)], ['both_correct', 'gap', 'both_wrong'], default='other')
    enriched_gap = mechanism_attribution_gap_cases[~mechanism_attribution_gap_cases.shot_id.isin(logical_population.shot_id)].copy()
    enriched_gap['cohort'] = 'gap'
    logical_population = pd.concat([logical_population, enriched_gap], ignore_index=True)
    logical_population = logical_population[logical_population.cohort != 'other'].drop_duplicates('shot_id').groupby(['distance', 'cohort'], sort=False).head(mechanism_attribution_config['logical_cost_case_limit_per_group'])

    for mechanism_attribution_case in tqdm(logical_population.itertuples(index=False), total=len(logical_population), desc='Experiment 8', disable=mechanism_attribution_config['print_level'] == 1):
        mechanism_attribution_context, mechanism_attribution_syndrome, mechanism_attribution_truth, mechanism_attribution_error = material(mechanism_attribution_case)
        mechanism_attribution_costs = mechanism_attribution_exact_sector_costs(mechanism_attribution_context, mechanism_attribution_syndrome)
        true_label = mechanism_attribution_bits(mechanism_attribution_truth)
        mechanism_attribution_true_cost = next((row['cost'] for row in mechanism_attribution_costs if row['sector'] == true_label))
        mechanism_attribution_competitor = min((row['cost'] for row in mechanism_attribution_costs if row['sector'] != true_label))

        for mechanism_attribution_row in mechanism_attribution_costs:
            logical_rows.append({
                'shot_id': mechanism_attribution_case.shot_id,
                'distance': mechanism_attribution_case.distance,
                'p': mechanism_attribution_case.p,
                'cohort': mechanism_attribution_case.cohort,
                'true_sector': true_label,
                'logical_margin': mechanism_attribution_competitor - mechanism_attribution_true_cost,
                **mechanism_attribution_row,
            })

    for mechanism_attribution_case_index, mechanism_attribution_case in enumerate(tqdm(mechanism_attribution_gap_cases.itertuples(index=False), total=len(mechanism_attribution_gap_cases), desc='Experiment 9', disable=mechanism_attribution_config['print_level'] == 1)):
        mechanism_attribution_context, mechanism_attribution_syndrome, mechanism_attribution_truth, mechanism_attribution_error = material(mechanism_attribution_case)
        mechanism_attribution_baseline = mechanism_attribution_trace_bp(mechanism_attribution_context, mechanism_attribution_syndrome, mechanism_attribution_config['bp_standard_iterations'])
        baseline_endpoint = mechanism_attribution_baseline.first_valid if mechanism_attribution_baseline.first_valid is not None else mechanism_attribution_config['bp_standard_iterations']
        baseline_hard = mechanism_attribution_baseline.hard[baseline_endpoint]
        baseline_sector = mechanism_attribution_logical_label(mechanism_attribution_context, baseline_hard)
        baseline_rank = np.argsort(np.abs(mechanism_attribution_baseline.llr[baseline_endpoint]))
        priors = []

        for multiplier in mechanism_attribution_config['prior_decode_multipliers']:
            p_decode = float(np.clip(mechanism_attribution_context.p * multiplier, 1e-09, 1 - 1e-09))
            decoder_context = mechanism_attribution_context_for(mechanism_attribution_context.level, p_decode)
            assert np.array_equal(decoder_context.H, mechanism_attribution_context.H) and np.array_equal(decoder_context.L, mechanism_attribution_context.L)
            priors.append((f'global_{multiplier:g}', decoder_context.prior_llr.copy()))

        matched = float(np.mean(mechanism_attribution_context.priors))
        priors.append(('matched_uniform', np.full(mechanism_attribution_context.H.shape[1], np.log((1 - matched) / matched))))
        mechanism_attribution_rng = np.random.default_rng(mechanism_attribution_config['random_seed'] + mechanism_attribution_case_index)

        for perturbation in range(mechanism_attribution_config['prior_perturbations']):
            priors.append((f'variable_{perturbation}', mechanism_attribution_context.prior_llr * (1 + mechanism_attribution_rng.normal(0, mechanism_attribution_config['prior_variable_sigma'], len(mechanism_attribution_context.prior_llr)))))
        for mechanism_attribution_name, prior_llr in priors:
            mechanism_attribution_trace = mechanism_attribution_trace_bp(mechanism_attribution_context, mechanism_attribution_syndrome, mechanism_attribution_config['bp_standard_iterations'], prior_llr=prior_llr)
            mechanism_attribution_endpoint = mechanism_attribution_trace.first_valid if mechanism_attribution_trace.first_valid is not None else mechanism_attribution_config['bp_standard_iterations']
            mechanism_attribution_hard = mechanism_attribution_trace.hard[mechanism_attribution_endpoint]
            mechanism_attribution_sector = mechanism_attribution_logical_label(mechanism_attribution_context, mechanism_attribution_hard)
            rank = np.argsort(np.abs(mechanism_attribution_trace.llr[mechanism_attribution_endpoint]))
            q_required = 'unrepaired'

            for mechanism_attribution_order in mechanism_attribution_config['osd_orders'] + [mechanism_attribution_full_osd_order(mechanism_attribution_context)]:
                mechanism_attribution_correction, mechanism_attribution_ = mechanism_attribution_osd_decode(mechanism_attribution_context, mechanism_attribution_syndrome, mechanism_attribution_trace.llr[mechanism_attribution_endpoint], mechanism_attribution_order)
                if mechanism_attribution_outcome(mechanism_attribution_context, mechanism_attribution_correction, mechanism_attribution_syndrome, mechanism_attribution_truth)[1]:
                    q_required = str(mechanism_attribution_order)
                    break

            prior_rows.append({
                'shot_id': mechanism_attribution_case.shot_id,
                'perturbation': mechanism_attribution_name,
                'syndrome_consistent': int(mechanism_attribution_trace.valid[mechanism_attribution_endpoint]),
                'logical_correct': mechanism_attribution_outcome(mechanism_attribution_context, mechanism_attribution_hard, mechanism_attribution_syndrome, mechanism_attribution_truth)[1],
                'logical_sector': mechanism_attribution_sector,
                'logical_switched': int(mechanism_attribution_sector != baseline_sector),
                'hard_switched': int(not np.array_equal(mechanism_attribution_hard, baseline_hard)),
                'rank_displacement': float(np.mean(np.abs(np.argsort(rank) - np.argsort(baseline_rank)))),
                'osd_order_required': q_required,
            })

    logical_costs = mechanism_attribution_save_csv(pd.DataFrame(logical_rows), LOGICAL_PATH)
    prior_trials = mechanism_attribution_save_csv(pd.DataFrame(prior_rows), PRIOR_PATH)

logical_margin_summary = logical_costs.drop_duplicates('shot_id').groupby(['distance', 'cohort'], as_index=False).agg(
    cases=('shot_id', 'size'),
    mean_margin=('logical_margin', 'mean'),
    median_abs_margin=('logical_margin', lambda x: np.median(np.abs(x))),
    exact_fraction=('exact', 'mean'),
)
mechanism_attribution_save_csv(logical_margin_summary, mechanism_attribution_data_paths.output('logical_margin_summary_csv'))
mechanism_attribution_save_csv(
    prior_trials.groupby('perturbation', as_index=False).agg(
        cases=('shot_id', 'nunique'),
        sector_switch_rate=('logical_switched', 'mean'),
        hard_switch_rate=('hard_switched', 'mean'),
        mean_rank_displacement=('rank_displacement', 'mean'),
    ),
    mechanism_attribution_data_paths.output('prior_sensitivity_summary_csv'),
)
logical_margin_summary


## Experiments 10–11: ambiguity-targeted rows and paired trapping subgraphs

Use ambiguity-informed redundant rows and inspect paired trapping subgraphs. The comparison asks whether the diagnostic region helps select a useful action, rather than merely producing a visually distinctive graph.


In [ ]:
TARGET_PATH = mechanism_attribution_trace_dir / 'experiment_10_targeted_rows.csv'
TRAP_PATH = mechanism_attribution_trace_dir / 'experiment_11_trapping_subgraphs.csv'

def variable_scores(context, trace, endpoint):
    history = trace.llr[max(0, endpoint - 4):endpoint + 1]
    instability = np.zeros(context.H.shape[1]) if len(history) < 2 else np.mean(np.abs(np.diff(history, axis=0)), axis=0)
    ambiguity = 1 / (np.abs(trace.llr[endpoint]) + 1e-09)
    conflict = np.zeros(context.H.shape[1])

    if endpoint == 0:
        return (instability, ambiguity, conflict)
    for variable, ids in enumerate(context.variable_edges):
        incoming = trace.c2v[max(1, endpoint - 3):endpoint + 1, ids]
        conflict[variable] = np.mean(1 - np.abs(incoming.sum(axis=1)) / (np.abs(incoming).sum(axis=1) + 1e-12))

    return (instability, ambiguity, conflict)

def targeted_transform(
    context,
    strategy,
    trace,
    endpoint,
    rng,
):
    candidates = redundant_row_candidates(context.H, mechanism_attribution_config['candidate_max_row_weight'])
    instability, ambiguity, conflict = variable_scores(context, trace, endpoint)
    scored = []

    for left, right, row in candidates:
        support = np.flatnonzero(row)
        values = {
            'random': rng.random(),
            'instability': instability[support].sum(),
            'ambiguity': ambiguity[support].sum(),
            'conflict': conflict[support].sum(),
        }

        values['hybrid'] = values['instability'] / (instability.sum() + 1e-12) + values['ambiguity'] / (ambiguity.sum() + 1e-12)
        scored.append((values[strategy], left, right, row))

    _, left, right, selected_row = max(scored, key=lambda item: item[0])
    transform_row = np.zeros(context.H.shape[0], dtype=np.uint8)
    transform_row[[left, right]] = 1

    return (np.vstack([np.eye(context.H.shape[0], dtype=np.uint8), transform_row]), selected_row)

def local_graph_metrics(
    context,
    H,
    variables,
    hops,
):
    assert hops in {1, 2}
    variables = set((int(value) for value in variables))
    checks, expanded_variables = (set(), set(variables))

    for _ in range(hops):
        checks |= set(np.flatnonzero(np.any(H[:, list(expanded_variables)], axis=1)).tolist()) if expanded_variables else set()
        expanded_variables |= set(np.flatnonzero(np.any(H[list(checks)], axis=0)).tolist()) if checks else set()

    sub_H = H[np.ix_(sorted(checks), sorted(expanded_variables))] if checks and expanded_variables else np.zeros((0, 0), dtype=np.uint8)
    if sub_H.size:
        four, six, girth, cycle_rank = cycle_counts(sub_H)
        average_degree = 2 * int(sub_H.sum()) / (sub_H.shape[0] + sub_H.shape[1])
    else:
        four, six, girth, cycle_rank, average_degree = (0, 0, np.nan, 0, 0.0)

    coords = context.detector_coordinates
    time_values = [coords.get(check, [np.nan])[-1] for check in checks]
    finite_times = [value for value in time_values if np.isfinite(value)]
    boundary = sum((H[:, variable].sum() <= 1 for variable in expanded_variables))

    return {
        'local_variables': len(expanded_variables),
        'local_checks': len(checks),
        'local_average_degree': average_degree,
        'local_cycle_rank': cycle_rank,
        'local_girth': girth,
        'local_four_cycles': four,
        'local_six_cycles': six,
        'boundary_involvement': boundary,
        'time_layer_crossing': int(len(set(finite_times)) > 1),
        'local_prior_heterogeneity': float(np.std(context.prior_llr[list(expanded_variables)])) if expanded_variables else 0.0,
    }

if mechanism_attribution_stage_cached([TARGET_PATH, TRAP_PATH]):
    targeted_rows = mechanism_attribution_read_csv(TARGET_PATH)
    trapping_pairs = mechanism_attribution_read_csv(TRAP_PATH)
else:
    target_rows, trap_rows = ([], [])
    for mechanism_attribution_case_index, mechanism_attribution_case in enumerate(tqdm(mechanism_attribution_gap_cases.itertuples(index=False), total=len(mechanism_attribution_gap_cases), desc='Experiments 10-11', disable=mechanism_attribution_config['print_level'] == 1)):
        mechanism_attribution_context, mechanism_attribution_syndrome, mechanism_attribution_truth, mechanism_attribution_error = material(mechanism_attribution_case)
        mechanism_attribution_baseline = mechanism_attribution_trace_bp(mechanism_attribution_context, mechanism_attribution_syndrome, mechanism_attribution_config['bp_standard_iterations'])
        mechanism_attribution_endpoint = mechanism_attribution_baseline.first_valid if mechanism_attribution_baseline.first_valid is not None else mechanism_attribution_config['bp_standard_iterations']
        mechanism_attribution_rng = np.random.default_rng(mechanism_attribution_config['random_seed'] + mechanism_attribution_case_index)
        mechanism_attribution_results = {}

        for mechanism_attribution_strategy in ['random', 'instability', 'ambiguity', 'conflict', 'hybrid']:
            mechanism_attribution_transform, selected_row = targeted_transform(mechanism_attribution_context, mechanism_attribution_strategy, mechanism_attribution_baseline, mechanism_attribution_endpoint, mechanism_attribution_rng)
            mechanism_attribution_view_context = mechanism_attribution_transformed_context(mechanism_attribution_context, mechanism_attribution_transform)
            mechanism_attribution_view_syndrome = mechanism_attribution_transform @ mechanism_attribution_syndrome % 2
            mechanism_attribution_trace = mechanism_attribution_trace_bp(mechanism_attribution_view_context, mechanism_attribution_view_syndrome, mechanism_attribution_config['bp_standard_iterations'])
            view_endpoint = mechanism_attribution_trace.first_valid if mechanism_attribution_trace.first_valid is not None else mechanism_attribution_config['bp_standard_iterations']
            mechanism_attribution_correction = mechanism_attribution_trace.hard[view_endpoint]
            mechanism_attribution_valid, mechanism_attribution_correct, mechanism_attribution_, mechanism_attribution_sector = mechanism_attribution_outcome(mechanism_attribution_context, mechanism_attribution_correction, mechanism_attribution_syndrome, mechanism_attribution_truth)
            mechanism_attribution_results[mechanism_attribution_strategy] = (mechanism_attribution_correct, mechanism_attribution_correction, mechanism_attribution_view_context)
            target_rows.append({
                'shot_id': mechanism_attribution_case.shot_id,
                'strategy': mechanism_attribution_strategy,
                'syndrome_consistent': mechanism_attribution_valid,
                'logical_correct': mechanism_attribution_correct,
                'logical_sector': mechanism_attribution_sector,
                'row_weight': int(selected_row.sum()),
                **mechanism_attribution_trace_metrics(mechanism_attribution_view_context, mechanism_attribution_trace, view_endpoint),
            })

        mechanism_attribution_successful = next(((name, values) for name, values in mechanism_attribution_results.items() if values[0]), None)
        if mechanism_attribution_successful is not None:
            fail_hard = mechanism_attribution_baseline.hard[mechanism_attribution_endpoint]
            repair_hard = mechanism_attribution_successful[1][1]
            mechanism_attribution_regions = {
                'least_reliable': np.argsort(np.abs(mechanism_attribution_baseline.llr[mechanism_attribution_endpoint]))[:mechanism_attribution_config['ambiguity_top_k']],
                'wrong_BP_support': np.flatnonzero(fail_hard ^ mechanism_attribution_error),
                'BP_to_repair_difference': np.flatnonzero(fail_hard ^ repair_hard),
            }

            for region_name, mechanism_attribution_variables in mechanism_attribution_regions.items():
                for mechanism_attribution_hops in [1, 2]:
                    for mechanism_attribution_role, H in [('failing', mechanism_attribution_context.H), ('repair', mechanism_attribution_successful[1][2].H)]:
                        trap_rows.append({
                            'shot_id': mechanism_attribution_case.shot_id,
                            'region': region_name,
                            'hops': mechanism_attribution_hops,
                            'role': mechanism_attribution_role,
                            'repair_strategy': mechanism_attribution_successful[0],
                            **local_graph_metrics(mechanism_attribution_context, H, mechanism_attribution_variables, mechanism_attribution_hops),
                        })

    targeted_rows = mechanism_attribution_save_csv(pd.DataFrame(target_rows), TARGET_PATH)
    trap_columns = ['shot_id', 'region', 'hops', 'role', 'repair_strategy', 'local_variables', 'local_checks', 'local_average_degree', 'local_cycle_rank', 'local_girth', 'local_four_cycles', 'local_six_cycles', 'boundary_involvement', 'time_layer_crossing', 'local_prior_heterogeneity']
    trapping_pairs = mechanism_attribution_save_csv(pd.DataFrame(trap_rows, columns=trap_columns), TRAP_PATH)

mechanism_attribution_save_csv(
    targeted_rows.groupby('strategy', as_index=False).agg(
        gap_cases=('shot_id', 'nunique'),
        repair_rate=('logical_correct', 'mean'),
        valid_rate=('syndrome_consistent', 'mean'),
        mean_instability=('message_instability', 'mean'),
    ),
    mechanism_attribution_data_paths.output('targeted_row_summary_csv'),
)
if len(trapping_pairs):
    trap_wide = trapping_pairs.pivot(index=['shot_id', 'region', 'hops'], columns='role')
    trap_differences = pd.DataFrame({column: trap_wide[column]['repair'] - trap_wide[column]['failing'] for column in ['local_variables', 'local_checks', 'local_average_degree', 'local_cycle_rank', 'local_four_cycles', 'local_six_cycles', 'boundary_involvement', 'time_layer_crossing', 'local_prior_heterogeneity']}).reset_index()
else:
    trap_differences = pd.DataFrame(columns=['shot_id', 'region', 'hops'])

mechanism_attribution_save_csv(trap_differences, mechanism_attribution_trace_dir / 'experiment_11_paired_subgraph_differences.csv')
if len(trap_differences):
    trap_aggregate = trap_differences.groupby(['region', 'hops'], as_index=False).mean(numeric_only=True).drop(columns='shot_id', errors='ignore')
else:
    trap_aggregate = pd.DataFrame(columns=['region', 'hops'])
mechanism_attribution_save_csv(trap_aggregate, mechanism_attribution_data_paths.output('paired_trapping_subgraph_summary_csv'))


## Experiments 12–13: mechanism attribution, overlap and conditional repair

Aggregate the intervention outcomes into mechanism frequencies and overlap. One case can respond to several interventions, so these labels are not a mutually exclusive causal partition.


In [ ]:
MECHANISM_PATH = mechanism_attribution_trace_dir / 'experiment_12_mechanism_matrix.csv'
if mechanism_attribution_stage_cached([MECHANISM_PATH]):
    mechanism_matrix = mechanism_attribution_read_csv(MECHANISM_PATH)
else:
    mechanism_matrix = mechanism_attribution_gap_cases[['shot_id', 'distance', 'p', 'split', 'message_instability', 'extrinsic_conflict']].copy()
    mechanism_matrix = mechanism_matrix.merge(early_stop[['shot_id', 'later_BP_correct']], on='shot_id')
    osd_wide = osd_orders.pivot(index='shot_id', columns='osd_label', values='logical_correct').reset_index()
    osd_wide = osd_wide.rename(columns={
        '0': 'OSD0_correct',
        '1': 'OSD1_correct',
        '2': 'OSD2_correct',
        'configured_full': 'full_OSD_correct',
    })
    mechanism_matrix = mechanism_matrix.merge(osd_wide, on='shot_id').merge(q_min[['shot_id', 'q_min']], on='shot_id')
    mechanism_matrix = mechanism_matrix.merge(pcm_summary, on='shot_id')
    update_wide = update_trials.pivot(index='shot_id', columns='variant', values='logical_correct')
    update_features = pd.DataFrame({
        'shot_id': update_wide.index,
        'sum_product_correct': update_wide.get('sum_product', pd.Series(0, index=update_wide.index)).to_numpy(),
        'scaled_BP_correct': update_wide[[column for column in update_wide if column.startswith('NMS_')]].max(axis=1).to_numpy(),
        'offset_BP_correct': update_wide[[column for column in update_wide if column.startswith('OMS_')]].max(axis=1).to_numpy(),
    }).reset_index(drop=True)
    mechanism_matrix = mechanism_matrix.merge(update_features, on='shot_id')
    mechanism_matrix = mechanism_matrix.merge(grid_repair_classes[['shot_id', 'scaling_repaired', 'damping_repaired', 'joint_repaired']], on='shot_id')
    mechanism_matrix = mechanism_matrix.rename(columns={'damping_repaired': 'damped_BP_correct'})
    restart_gap = restart_summary[restart_summary.pcm == 'standard_PCM'][['shot_id', 'restart_correct_fraction', 'restart_unique_sectors']]
    mechanism_matrix = mechanism_matrix.merge(restart_gap, on='shot_id', how='left')
    mechanism_attribution_margins = logical_costs.drop_duplicates('shot_id')[['shot_id', 'logical_margin']]
    mechanism_matrix = mechanism_matrix.merge(mechanism_attribution_margins, on='shot_id', how='left')
    prior_summary = prior_trials.groupby('shot_id', as_index=False).agg(prior_sensitivity=('logical_switched', 'mean'))
    mechanism_matrix = mechanism_matrix.merge(prior_summary, on='shot_id', how='left')
    mechanism_attribution_development_instability = mechanism_matrix.loc[mechanism_matrix.split == 'development', 'message_instability'].to_numpy()

    if mechanism_attribution_config['smoke_mode'] and (not len(mechanism_attribution_development_instability)):
        mechanism_attribution_development_instability = mechanism_matrix.message_instability.to_numpy()
    assert len(mechanism_attribution_development_instability), 'No development instability values for mechanism thresholds'
    mechanism_attribution_development_instability = np.sort(mechanism_attribution_development_instability)
    mechanism_matrix['instability_level'] = np.searchsorted(mechanism_attribution_development_instability, mechanism_matrix.message_instability.to_numpy(), side='right') / len(mechanism_attribution_development_instability)
    mechanism_matrix['ambiguity_level'] = 1 / (mechanism_matrix.logical_margin.abs() + 1e-09)
    high_instability = mechanism_matrix.instability_level >= 0.75
    mechanism_matrix['early_stop'] = mechanism_matrix.later_BP_correct.astype(int)
    mechanism_matrix['hard_decision_soft_info'] = ((mechanism_matrix.later_BP_correct == 0) & (mechanism_matrix.OSD1_correct == 1)).astype(int)
    mechanism_matrix['pcm_sensitive'] = mechanism_matrix.PCM_any_correct.astype(int)
    mechanism_matrix['update_rule_sensitive'] = mechanism_matrix.sum_product_correct.astype(int)
    mechanism_matrix['restart_multiple_sectors'] = (mechanism_matrix.restart_unique_sectors.fillna(1) > 1).astype(int)
    mechanism_matrix['multi_basin'] = mechanism_matrix.restart_multiple_sectors
    mechanism_matrix['oscillation_sensitive'] = ((mechanism_matrix.damped_BP_correct == 1) & high_instability).astype(int)
    mechanism_matrix['logical_ambiguity'] = (mechanism_matrix.logical_margin.abs() < mechanism_attribution_config['logical_margin_threshold']).fillna(False).astype(int)
    base_search_limited = (mechanism_matrix.later_BP_correct == 0) & (mechanism_matrix.PCM_any_correct == 0) & (mechanism_matrix.sum_product_correct == 0) & (mechanism_matrix.damped_BP_correct == 0) & (mechanism_matrix.scaling_repaired == 0) & (mechanism_matrix.offset_BP_correct == 0) & (mechanism_matrix.restart_correct_fraction.fillna(0) == 0) & (mechanism_matrix.OSD1_correct == 0) & (mechanism_matrix.OSD2_correct == 0)
    mechanism_matrix['search_limited_full_osd'] = (base_search_limited & (mechanism_matrix.full_OSD_correct == 1)).astype(int)
    mechanism_matrix['search_limited_tesseract_only'] = (base_search_limited & (mechanism_matrix.full_OSD_correct == 0)).astype(int)
    mechanism_matrix['search_limited'] = (mechanism_matrix.search_limited_full_osd | mechanism_matrix.search_limited_tesseract_only).astype(int)
    mechanism_matrix = mechanism_attribution_save_csv(mechanism_matrix, MECHANISM_PATH)

LABELS = ['early_stop', 'hard_decision_soft_info', 'pcm_sensitive', 'update_rule_sensitive', 'multi_basin', 'oscillation_sensitive', 'logical_ambiguity', 'search_limited']
label_summary = pd.DataFrame({
    'mechanism': LABELS,
    'gap_cases': len(mechanism_matrix),
    'fraction': [mechanism_matrix[label].mean() for label in LABELS],
})
mechanism_attribution_save_csv(label_summary, mechanism_attribution_data_paths.output('mechanism_frequency_csv'))
overlap = mechanism_matrix[LABELS].astype(int).astype(str).agg(''.join, axis=1).value_counts().rename_axis('label_pattern').reset_index(name='cases')
mechanism_attribution_save_csv(overlap, mechanism_attribution_data_paths.output('mechanism_overlap_csv'))
interventions = {
    'Continue_BP': 'later_BP_correct',
    'Damping': 'damped_BP_correct',
    'Sum_product': 'sum_product_correct',
    'PCM_Ensemble': 'PCM_any_correct',
    'OSD1': 'OSD1_correct',
}
conditional_rows = []

for mechanism_attribution_split_name in ['all', 'development', 'test']:
    mechanism_attribution_source = mechanism_matrix if mechanism_attribution_split_name == 'all' else mechanism_matrix[mechanism_matrix.split == mechanism_attribution_split_name]
    for label in LABELS:
        mechanism_attribution_subset = mechanism_attribution_source[mechanism_attribution_source[label] == 1]
        for intervention, mechanism_attribution_column in interventions.items():
            conditional_rows.append({
                'split': mechanism_attribution_split_name,
                'mechanism': label,
                'intervention': intervention,
                'cases': len(mechanism_attribution_subset),
                'repair_rate': np.nan if not len(mechanism_attribution_subset) else mechanism_attribution_subset[mechanism_attribution_column].mean(),
            })

mechanism_repair = mechanism_attribution_save_csv(pd.DataFrame(conditional_rows), mechanism_attribution_data_paths.output('mechanism_conditional_repair_csv'))
mechanism_matrix.head()


## Targeted Repair A–E and final mechanism-aware decoder

Evaluate the targeted repair variants and assemble the tested mechanism-aware rule. Compare candidate availability with the rule's ability to choose a correction without truth.


In [ ]:
TARGETED_PATH = mechanism_attribution_trace_dir / 'targeted_repair_trials.csv'
POLICY_PATH = mechanism_attribution_data_paths.output('mechanism_aware_policy_csv')
mechanism_attribution_development = unbiased_population[(unbiased_population.split == 'development') & unbiased_population.distance.isin(mechanism_attribution_primary_distances)]
policy_mechanisms = mechanism_matrix[mechanism_matrix.split == 'development']

if mechanism_attribution_config['smoke_mode'] and policy_mechanisms.empty:
    policy_mechanisms = mechanism_matrix
assert len(policy_mechanisms), 'No development gap cases for mechanism-aware policy selection'
label_rates = policy_mechanisms[LABELS].mean()

if label_rates.max() > 0:
    dominant_mechanism = label_rates.idxmax()
    policy_split = 'development' if not mechanism_repair[(mechanism_repair.split == 'development') & (mechanism_repair.mechanism == dominant_mechanism) & (mechanism_repair.cases > 0)].empty else 'all'
    policy_repairs = mechanism_repair[(mechanism_repair.split == policy_split) & (mechanism_repair.mechanism == dominant_mechanism) & (mechanism_repair.cases > 0)]
    preferred_intervention = policy_repairs.sort_values(['repair_rate', 'intervention'], ascending=[False, True]).iloc[0].intervention
else:
    dominant_mechanism = 'unattributed'
    policy_split = 'development' if not mechanism_matrix[mechanism_matrix.split == 'development'].empty else 'all'
    policy_source = mechanism_matrix if policy_split == 'all' else mechanism_matrix[mechanism_matrix.split == policy_split]
    preferred_intervention = max(interventions, key=lambda name: (policy_source[interventions[name]].mean(), name))

mechanism_attribution_policy = pd.DataFrame([{
    'conflict_threshold': float(mechanism_attribution_development.extrinsic_conflict.quantile(0.75)),
    'instability_threshold': float(mechanism_attribution_development.message_instability.quantile(0.75)),
    'ambiguity_abs_llr_threshold': float(mechanism_attribution_development.posterior_abs_mean.quantile(0.25)),
    'adaptive_max_views': mechanism_attribution_config['adaptive_max_views'],
    'dominant_mechanism': dominant_mechanism,
    'preferred_intervention': preferred_intervention,
    'selection_split': policy_split,
}])
mechanism_attribution_save_csv(mechanism_attribution_policy, POLICY_PATH)

def collect_candidate(
    context,
    syndrome,
    truth,
    correction,
    name,
    runtime,
    bp_runs,
    pcm_views_used,
    iterations,
    osd_order=-1,
):
    valid, correct, _, sector = mechanism_attribution_outcome(context, correction, syndrome, truth)
    return {
        'method': name,
        'syndrome_consistent': valid,
        'logical_correct': correct,
        'logical_sector': sector,
        'prior_cost': mechanism_attribution_prior_cost(context, correction) if valid else np.inf,
        'runtime': runtime,
        'bp_runs': bp_runs,
        'pcm_views': pcm_views_used,
        'bp_iterations': iterations,
        'runtime_scope': 'single_shot_bp_osd_kernel',
        'osd1_used': int(osd_order == 1),
        'full_osd_used': int(osd_order > 1),
        'correction': correction,
    }

def pcm_ensemble_decode(
    context,
    syndrome,
    truth_unused,
    baseline,
    strategy,
    views,
    seed,
):
    valid = np.flatnonzero(baseline.valid[1:mechanism_attribution_config['bp_standard_iterations'] + 1]) + 1
    endpoint = int(valid[0]) if len(valid) else mechanism_attribution_config['bp_standard_iterations']
    rng = np.random.default_rng(seed)
    candidates = []
    total_runtime = 0.0
    total_iterations = 0
    baseline_correction = baseline.hard[endpoint]

    if np.array_equal(context.H @ baseline_correction % 2, syndrome):
        candidates.append((mechanism_attribution_prior_cost(context, baseline_correction), baseline_correction, baseline.llr[endpoint]))
    for view_index in range(views):
        if strategy == 'same_PCM_restart':
            transform = np.eye(context.H.shape[0], dtype=np.uint8)
            trace = mechanism_attribution_trace_bp(context, syndrome, mechanism_attribution_config['bp_standard_iterations'], rng=rng, initial_message_sigma=mechanism_attribution_config['restart_message_sigma'])
            view_context, view_syndrome = (context, syndrome)
        elif strategy == 'random_PCM':
            transform = random_pcm_views(context, 1, rng)[-1].transform
            view_context = mechanism_attribution_transformed_context(context, transform)
            view_syndrome = transform @ syndrome % 2
            trace = mechanism_attribution_trace_bp(view_context, view_syndrome, mechanism_attribution_config['bp_standard_iterations'])
        else:
            target = 'instability' if strategy == 'instability_PCM' else 'ambiguity'
            transform, _ = targeted_transform(context, target, baseline, endpoint, rng)
            view_context = mechanism_attribution_transformed_context(context, transform)
            view_syndrome = transform @ syndrome % 2
            trace = mechanism_attribution_trace_bp(view_context, view_syndrome, mechanism_attribution_config['bp_standard_iterations'])

        view_valid = np.flatnonzero(trace.valid[1:mechanism_attribution_config['bp_standard_iterations'] + 1]) + 1
        view_endpoint = int(view_valid[0]) if len(view_valid) else mechanism_attribution_config['bp_standard_iterations']
        correction = trace.hard[view_endpoint]
        total_runtime += trace.runtime[view_endpoint]
        total_iterations += view_endpoint

        if np.array_equal(context.H @ correction % 2, syndrome):
            candidates.append((mechanism_attribution_prior_cost(context, correction), correction, trace.llr[view_endpoint]))

    if not candidates:
        return (baseline.hard[endpoint], baseline.llr[endpoint], total_runtime, total_iterations)
    best = min(candidates, key=lambda item: item[0])

    return (best[1], best[2], total_runtime, total_iterations)

if mechanism_attribution_stage_cached([TARGETED_PATH]):
    targeted_trials = mechanism_attribution_read_csv(TARGETED_PATH)
else:
    mechanism_attribution_rows = []
    for mechanism_attribution_case_index, mechanism_attribution_case in enumerate(tqdm(mechanism_attribution_gap_cases.itertuples(index=False), total=len(mechanism_attribution_gap_cases), desc='Targeted Repair A-E', disable=mechanism_attribution_config['print_level'] == 1)):
        mechanism_attribution_context, mechanism_attribution_syndrome, mechanism_attribution_truth, mechanism_attribution_error = material(mechanism_attribution_case)
        mechanism_attribution_baseline = mechanism_attribution_trace_bp(mechanism_attribution_context, mechanism_attribution_syndrome, mechanism_attribution_config['continued_iterations'])
        mechanism_attribution_endpoint = next(iter(np.flatnonzero(mechanism_attribution_baseline.valid[1:mechanism_attribution_config['bp_standard_iterations'] + 1]) + 1), mechanism_attribution_config['bp_standard_iterations'])
        standard_correction, mechanism_attribution_, standard_osd, mechanism_attribution_standard_runtime = mechanism_attribution_standard_bposd(mechanism_attribution_context, mechanism_attribution_syndrome, mechanism_attribution_baseline)
        base = collect_candidate(mechanism_attribution_context, mechanism_attribution_syndrome, mechanism_attribution_truth, standard_correction, 'standard_BPOSD', mechanism_attribution_standard_runtime, 1, 1, mechanism_attribution_endpoint, mechanism_attribution_full_osd_order(mechanism_attribution_context) if standard_osd else -1)
        base.update(shot_id=mechanism_attribution_case.shot_id, strategy_family='baseline', view_budget=1)
        mechanism_attribution_rows.append({k: v for k, v in base.items() if k != 'correction'})
        suspicious = not bool(mechanism_attribution_case.standard_bp_valid) or mechanism_attribution_case.message_instability >= mechanism_attribution_policy.iloc[0].instability_threshold or mechanism_attribution_case.extrinsic_conflict >= mechanism_attribution_policy.iloc[0].conflict_threshold or (mechanism_attribution_case.posterior_abs_mean <= mechanism_attribution_policy.iloc[0].ambiguity_abs_llr_threshold)
        adaptive = mechanism_attribution_baseline.hard[mechanism_attribution_endpoint]
        adaptive_runtime = mechanism_attribution_baseline.runtime[mechanism_attribution_endpoint]
        adaptive_runs = 1
        adaptive_iterations = mechanism_attribution_endpoint

        if suspicious:
            adaptive, mechanism_attribution_, extra, extra_iterations = pcm_ensemble_decode(mechanism_attribution_context, mechanism_attribution_syndrome, mechanism_attribution_truth, mechanism_attribution_baseline, 'ambiguity_PCM', mechanism_attribution_config['adaptive_max_views'], mechanism_attribution_config['random_seed'] + mechanism_attribution_case_index)
            adaptive_runtime += extra
            adaptive_iterations += extra_iterations
            adaptive_runs += mechanism_attribution_config['adaptive_max_views']

        adaptive_result = collect_candidate(mechanism_attribution_context, mechanism_attribution_syndrome, mechanism_attribution_truth, adaptive, 'adaptive_PCM_switching', adaptive_runtime, adaptive_runs, adaptive_runs, adaptive_iterations)
        adaptive_result.update(shot_id=mechanism_attribution_case.shot_id, strategy_family='adaptive_PCM_switching', view_budget=mechanism_attribution_config['adaptive_max_views'])
        mechanism_attribution_rows.append({k: v for k, v in adaptive_result.items() if k != 'correction'})

        for mechanism_attribution_views in mechanism_attribution_config['pcm_ensemble_sizes']:
            for mechanism_attribution_strategy in ['same_PCM_restart', 'random_PCM', 'instability_PCM', 'ambiguity_PCM']:
                mechanism_attribution_correction, mechanism_attribution_, mechanism_attribution_runtime, run_iterations = pcm_ensemble_decode(mechanism_attribution_context, mechanism_attribution_syndrome, mechanism_attribution_truth, mechanism_attribution_baseline, mechanism_attribution_strategy, mechanism_attribution_views, mechanism_attribution_config['random_seed'] + mechanism_attribution_case_index * 100 + mechanism_attribution_views)
                result = collect_candidate(mechanism_attribution_context, mechanism_attribution_syndrome, mechanism_attribution_truth, mechanism_attribution_correction, f'{mechanism_attribution_strategy}_M{mechanism_attribution_views}', mechanism_attribution_baseline.runtime[mechanism_attribution_endpoint] + mechanism_attribution_runtime, 1 + mechanism_attribution_views, 1 + mechanism_attribution_views, mechanism_attribution_endpoint + run_iterations)
                result.update(shot_id=mechanism_attribution_case.shot_id, strategy_family=mechanism_attribution_strategy, view_budget=mechanism_attribution_views)
                mechanism_attribution_rows.append({k: v for k, v in result.items() if k != 'correction'})

        sum_trace = mechanism_attribution_trace_bp(mechanism_attribution_context, mechanism_attribution_syndrome, mechanism_attribution_config['bp_standard_iterations'], update_rule='sum_product')
        sum_endpoint = sum_trace.first_valid if sum_trace.first_valid is not None else mechanism_attribution_config['bp_standard_iterations']
        update_result = collect_candidate(mechanism_attribution_context, mechanism_attribution_syndrome, mechanism_attribution_truth, sum_trace.hard[sum_endpoint], 'adaptive_update_rule', sum_trace.runtime[sum_endpoint], 2, 1, mechanism_attribution_endpoint + sum_endpoint)
        update_result.update(shot_id=mechanism_attribution_case.shot_id, strategy_family='update_rule_correction', view_budget=1)
        mechanism_attribution_rows.append({k: v for k, v in update_result.items() if k != 'correction'})
        osd1, osd_runtime = mechanism_attribution_osd_decode(mechanism_attribution_context, mechanism_attribution_syndrome, mechanism_attribution_baseline.llr[mechanism_attribution_endpoint], 1)
        osd_result = collect_candidate(mechanism_attribution_context, mechanism_attribution_syndrome, mechanism_attribution_truth, osd1, 'BP_plus_OSD1', mechanism_attribution_baseline.runtime[mechanism_attribution_endpoint] + osd_runtime, 1, 1, mechanism_attribution_endpoint, 1)
        osd_result.update(shot_id=mechanism_attribution_case.shot_id, strategy_family='low_order_OSD', view_budget=1)
        mechanism_attribution_rows.append({k: v for k, v in osd_result.items() if k != 'correction'})

    targeted_trials = mechanism_attribution_save_csv(pd.DataFrame(mechanism_attribution_rows), TARGETED_PATH)

targeted_summary = targeted_trials.groupby(['strategy_family', 'view_budget'], as_index=False).agg(
    gap_cases=('shot_id', 'nunique'),
    conditional_repair=('logical_correct', 'mean'),
    valid_rate=('syndrome_consistent', 'mean'),
    mean_bp_runs=('bp_runs', 'mean'),
    mean_runtime=('runtime', 'mean'),
)
mechanism_attribution_save_csv(targeted_summary, mechanism_attribution_data_paths.output('targeted_repair_summary_csv'))
targeted_summary


## Final paired cross-p and cross-distance benchmark

Evaluate the final rules on paired physical shots across the configured p values and distances. Include baseline successes so regressions remain visible alongside repairs.


In [ ]:
FINAL_PATH = mechanism_attribution_trace_dir / 'final_benchmark_trials.csv'
if mechanism_attribution_stage_cached([FINAL_PATH]):
    mechanism_attribution_final_trials = mechanism_attribution_read_csv(FINAL_PATH)
else:
    test_population = unbiased_population[unbiased_population.split == 'test'].copy()
    if mechanism_attribution_config['final_case_limit_per_point'] > 0:
        test_population = test_population.groupby(['distance', 'p'], sort=False).head(mechanism_attribution_config['final_case_limit_per_point'])
    mechanism_attribution_rows = []
    mechanism_attribution_thresholds = mechanism_attribution_policy.iloc[0]

    for mechanism_attribution_case_index, mechanism_attribution_case in enumerate(tqdm(test_population.itertuples(index=False), total=len(test_population), desc='Final benchmark', disable=mechanism_attribution_config['print_level'] == 1)):
        mechanism_attribution_context, mechanism_attribution_syndrome, mechanism_attribution_truth, mechanism_attribution_error = material(mechanism_attribution_case)
        mechanism_attribution_trace = mechanism_attribution_trace_bp(mechanism_attribution_context, mechanism_attribution_syndrome, mechanism_attribution_config['continued_iterations'])
        mechanism_attribution_standard, mechanism_attribution_endpoint, standard_osd, mechanism_attribution_standard_runtime = mechanism_attribution_standard_bposd(mechanism_attribution_context, mechanism_attribution_syndrome, mechanism_attribution_trace)
        standard_bp_valid = np.flatnonzero(mechanism_attribution_trace.valid[1:mechanism_attribution_config['bp_standard_iterations'] + 1]) + 1
        standard_bp_endpoint = int(standard_bp_valid[0]) if len(standard_bp_valid) else mechanism_attribution_config['bp_standard_iterations']
        mechanism_attribution_methods = [collect_candidate(mechanism_attribution_context, mechanism_attribution_syndrome, mechanism_attribution_truth, mechanism_attribution_trace.hard[standard_bp_endpoint], 'standard_BP', mechanism_attribution_trace.runtime[standard_bp_endpoint], 1, 1, standard_bp_endpoint), collect_candidate(mechanism_attribution_context, mechanism_attribution_syndrome, mechanism_attribution_truth, mechanism_attribution_standard, 'standard_BPOSD', mechanism_attribution_standard_runtime, 1, 1, mechanism_attribution_endpoint, mechanism_attribution_full_osd_order(mechanism_attribution_context) if standard_osd else -1)]
        later_valid = [t for t in np.flatnonzero(mechanism_attribution_trace.valid) if t > standard_bp_endpoint]
        continued_endpoint = int(later_valid[-1]) if later_valid else mechanism_attribution_config['continued_iterations']
        continued = mechanism_attribution_trace.hard[continued_endpoint]
        mechanism_attribution_methods.append(collect_candidate(mechanism_attribution_context, mechanism_attribution_syndrome, mechanism_attribution_truth, continued, 'continued_BP', mechanism_attribution_trace.runtime[continued_endpoint], 1, 1, continued_endpoint))
        sum_trace = mechanism_attribution_trace_bp(mechanism_attribution_context, mechanism_attribution_syndrome, mechanism_attribution_config['bp_standard_iterations'], update_rule='sum_product')
        sum_endpoint = sum_trace.first_valid if sum_trace.first_valid is not None else mechanism_attribution_config['bp_standard_iterations']
        mechanism_attribution_methods.append(collect_candidate(mechanism_attribution_context, mechanism_attribution_syndrome, mechanism_attribution_truth, sum_trace.hard[sum_endpoint], 'sum_product_BP', sum_trace.runtime[sum_endpoint], 1, 1, sum_endpoint))
        random_pcm, mechanism_attribution_, random_runtime, random_iterations = pcm_ensemble_decode(mechanism_attribution_context, mechanism_attribution_syndrome, mechanism_attribution_truth, mechanism_attribution_trace, 'random_PCM', max(mechanism_attribution_config['pcm_ensemble_sizes']), mechanism_attribution_config['random_seed'] + mechanism_attribution_case_index)
        mechanism_attribution_methods.append(collect_candidate(mechanism_attribution_context, mechanism_attribution_syndrome, mechanism_attribution_truth, random_pcm, 'PCM_ensemble_BP', mechanism_attribution_trace.runtime[standard_bp_endpoint] + random_runtime, 1 + max(mechanism_attribution_config['pcm_ensemble_sizes']), 1 + max(mechanism_attribution_config['pcm_ensemble_sizes']), standard_bp_endpoint + random_iterations))
        ambiguity_pcm, mechanism_attribution_, ambiguity_runtime, ambiguity_iterations = pcm_ensemble_decode(mechanism_attribution_context, mechanism_attribution_syndrome, mechanism_attribution_truth, mechanism_attribution_trace, 'ambiguity_PCM', mechanism_attribution_config['adaptive_max_views'], mechanism_attribution_config['random_seed'] + mechanism_attribution_case_index)
        mechanism_attribution_methods.append(collect_candidate(mechanism_attribution_context, mechanism_attribution_syndrome, mechanism_attribution_truth, ambiguity_pcm, 'ambiguity_targeted_PCM_BP', mechanism_attribution_trace.runtime[standard_bp_endpoint] + ambiguity_runtime, 1 + mechanism_attribution_config['adaptive_max_views'], 1 + mechanism_attribution_config['adaptive_max_views'], standard_bp_endpoint + ambiguity_iterations))
        suspicious = not bool(mechanism_attribution_case.standard_bp_valid) or mechanism_attribution_case.message_instability >= mechanism_attribution_thresholds.instability_threshold or mechanism_attribution_case.extrinsic_conflict >= mechanism_attribution_thresholds.conflict_threshold or (mechanism_attribution_case.posterior_abs_mean <= mechanism_attribution_thresholds.ambiguity_abs_llr_threshold)
        aware = mechanism_attribution_trace.hard[standard_bp_endpoint]
        aware_llr = mechanism_attribution_trace.llr[standard_bp_endpoint]
        aware_runtime = mechanism_attribution_trace.runtime[standard_bp_endpoint]
        aware_runs = 1
        aware_views = 1
        aware_order = -1
        aware_iterations = standard_bp_endpoint

        if suspicious:
            if mechanism_attribution_thresholds.preferred_intervention == 'Continue_BP':
                aware = continued
                aware_llr = mechanism_attribution_trace.llr[continued_endpoint]
                aware_runtime = mechanism_attribution_trace.runtime[continued_endpoint]
                aware_iterations = continued_endpoint
            elif mechanism_attribution_thresholds.preferred_intervention == 'Damping':
                corrected_trace = mechanism_attribution_trace_bp(mechanism_attribution_context, mechanism_attribution_syndrome, mechanism_attribution_config['bp_standard_iterations'], damping=0.5)
                corrected_endpoint = corrected_trace.first_valid if corrected_trace.first_valid is not None else mechanism_attribution_config['bp_standard_iterations']
                aware = corrected_trace.hard[corrected_endpoint]
                aware_llr = corrected_trace.llr[corrected_endpoint]
                aware_runtime += corrected_trace.runtime[corrected_endpoint]
                aware_runs += 1
                aware_iterations += corrected_endpoint
            elif mechanism_attribution_thresholds.preferred_intervention == 'Sum_product':
                aware = sum_trace.hard[sum_endpoint]
                aware_llr = sum_trace.llr[sum_endpoint]
                aware_runtime += sum_trace.runtime[sum_endpoint]
                aware_runs += 1
                aware_iterations += sum_endpoint
            elif mechanism_attribution_thresholds.preferred_intervention == 'OSD1':
                aware, extra = mechanism_attribution_osd_decode(mechanism_attribution_context, mechanism_attribution_syndrome, aware_llr, 1)
                aware_runtime += extra
                aware_order = 1
            else:
                assert mechanism_attribution_thresholds.preferred_intervention == 'PCM_Ensemble'
                aware, aware_llr, extra, extra_iterations = pcm_ensemble_decode(mechanism_attribution_context, mechanism_attribution_syndrome, mechanism_attribution_truth, mechanism_attribution_trace, 'ambiguity_PCM', mechanism_attribution_config['adaptive_max_views'], mechanism_attribution_config['random_seed'] + mechanism_attribution_case_index)
                aware_runtime += extra
                aware_iterations += extra_iterations
                aware_runs += mechanism_attribution_config['adaptive_max_views']
                aware_views += mechanism_attribution_config['adaptive_max_views']

        mechanism_attribution_methods.append(collect_candidate(mechanism_attribution_context, mechanism_attribution_syndrome, mechanism_attribution_truth, aware, 'mechanism_aware_BP', aware_runtime, aware_runs, aware_views, aware_iterations, aware_order))
        aware_osd = aware
        aware_osd_runtime = aware_runtime
        aware_osd_order = aware_order

        if aware_osd_order != 1:
            aware_osd, extra = mechanism_attribution_osd_decode(mechanism_attribution_context, mechanism_attribution_syndrome, aware_llr, 1)
            aware_osd_runtime += extra
            aware_osd_order = 1
        mechanism_attribution_methods.append(collect_candidate(mechanism_attribution_context, mechanism_attribution_syndrome, mechanism_attribution_truth, aware_osd, 'mechanism_aware_BP_plus_OSD1', aware_osd_runtime, aware_runs, aware_views, aware_iterations, aware_osd_order))
        mechanism_attribution_forced, mechanism_attribution_forced_runtime = mechanism_attribution_osd_decode(mechanism_attribution_context, mechanism_attribution_syndrome, mechanism_attribution_trace.llr[standard_bp_endpoint], mechanism_attribution_full_osd_order(mechanism_attribution_context))
        mechanism_attribution_methods.append(collect_candidate(mechanism_attribution_context, mechanism_attribution_syndrome, mechanism_attribution_truth, mechanism_attribution_forced, 'forced_configured_full_OSD', mechanism_attribution_trace.runtime[standard_bp_endpoint] + mechanism_attribution_forced_runtime, 1, 1, standard_bp_endpoint, mechanism_attribution_full_osd_order(mechanism_attribution_context)))
        mechanism_attribution_methods.append({
            'method': 'Tesseract',
            'syndrome_consistent': np.nan,
            'logical_correct': int(mechanism_attribution_case.tesseract_correct),
            'logical_sector': mechanism_attribution_case.tesseract_sector,
            'prior_cost': np.nan,
            'runtime': mechanism_attribution_case.tesseract_runtime,
            'bp_runs': 0,
            'pcm_views': 0,
            'bp_iterations': 0,
            'osd1_used': 0,
            'full_osd_used': 0,
            'runtime_scope': 'single_shot_decode_plus_amortized_compile',
            'correction': np.zeros(mechanism_attribution_context.H.shape[1], dtype=np.uint8),
        })

        for result in mechanism_attribution_methods:
            result.update(
                shot_id=mechanism_attribution_case.shot_id,
                distance=mechanism_attribution_case.distance,
                p=mechanism_attribution_case.p,
                standard_correct=int(mechanism_attribution_case.standard_correct),
                tesseract_correct=int(mechanism_attribution_case.tesseract_correct),
            )
            mechanism_attribution_rows.append({k: v for k, v in result.items() if k != 'correction'})

    mechanism_attribution_final_trials = mechanism_attribution_save_csv(pd.DataFrame(mechanism_attribution_rows), FINAL_PATH)

mechanism_attribution_final_summary = mechanism_attribution_final_trials.groupby(['distance', 'p', 'method', 'runtime_scope'], as_index=False).agg(
    shots=('shot_id', 'nunique'),
    LER=('logical_correct', lambda x: 1 - x.mean()),
    syndrome_consistent_rate=('syndrome_consistent', 'mean'),
    mean_bp_runs=('bp_runs', 'mean'),
    mean_pcm_views=('pcm_views', 'mean'),
    mean_bp_iterations=('bp_iterations', 'mean'),
    fraction_OSD1=('osd1_used', 'mean'),
    fraction_full_OSD=('full_osd_used', 'mean'),
    runtime_samples=('runtime', 'count'),
    mean_runtime=('runtime', 'mean'),
    p95_runtime=('runtime', lambda x: x.quantile(0.95)),
)
mechanism_attribution_reference = mechanism_attribution_final_summary.pivot(index=['distance', 'p'], columns='method', values='LER')
gap_rows = []

for mechanism_attribution_row in mechanism_attribution_final_summary.itertuples(index=False):
    mechanism_attribution_baseline = mechanism_attribution_reference.loc[(mechanism_attribution_row.distance, mechanism_attribution_row.p), 'standard_BPOSD']
    mechanism_attribution_tess = mechanism_attribution_reference.loc[(mechanism_attribution_row.distance, mechanism_attribution_row.p), 'Tesseract']
    mechanism_attribution_denominator = mechanism_attribution_baseline - mechanism_attribution_tess
    trials = mechanism_attribution_final_trials[(mechanism_attribution_final_trials.distance == mechanism_attribution_row.distance) & (mechanism_attribution_final_trials.p == mechanism_attribution_row.p)]
    mechanism_attribution_method = trials[trials.method == mechanism_attribution_row.method].set_index('shot_id')
    mechanism_attribution_standard = trials[trials.method == 'standard_BPOSD'].set_index('shot_id')
    common = mechanism_attribution_method.index.intersection(mechanism_attribution_standard.index)
    mechanism_attribution_gap_mask = (mechanism_attribution_standard.loc[common].logical_correct == 0) & (mechanism_attribution_standard.loc[common].tesseract_correct == 1)
    regress_mask = mechanism_attribution_standard.loc[common].logical_correct == 1
    gap_rows.append({
        'distance': mechanism_attribution_row.distance,
        'p': mechanism_attribution_row.p,
        'method': mechanism_attribution_row.method,
        'gap_closure': np.nan if np.isclose(mechanism_attribution_denominator, 0) else (mechanism_attribution_baseline - mechanism_attribution_row.LER) / mechanism_attribution_denominator,
        'conditional_gap_repair': np.nan if not mechanism_attribution_gap_mask.any() else mechanism_attribution_method.loc[common].loc[mechanism_attribution_gap_mask].logical_correct.mean(),
        'regression_rate': np.nan if not regress_mask.any() else 1 - mechanism_attribution_method.loc[common].loc[regress_mask].logical_correct.mean(),
    })

mechanism_attribution_final_summary = mechanism_attribution_final_summary.merge(pd.DataFrame(gap_rows), on=['distance', 'p', 'method'])
mechanism_attribution_save_csv(mechanism_attribution_final_summary, mechanism_attribution_data_paths.output('final_benchmark_csv'))
mechanism_attribution_bootstrap_rows = []

for (mechanism_attribution_distance, mechanism_attribution_p), mechanism_attribution_point in mechanism_attribution_final_trials.groupby(['distance', 'p']):
    mechanism_attribution_wide = mechanism_attribution_point.pivot(index='shot_id', columns='method', values='logical_correct').sort_index()
    assert not mechanism_attribution_wide.isna().any().any()
    mechanism_attribution_rng = np.random.default_rng(mechanism_attribution_config['random_seed'] + int(mechanism_attribution_distance * 1000 + mechanism_attribution_p * 1000000.0))
    mechanism_attribution_indices = mechanism_attribution_rng.integers(0, len(mechanism_attribution_wide), size=(mechanism_attribution_config['bootstrap_repetitions'], len(mechanism_attribution_wide)))
    mechanism_attribution_baseline_error = 1 - mechanism_attribution_wide['standard_BPOSD'].to_numpy(dtype=float)
    tess_error = 1 - mechanism_attribution_wide['Tesseract'].to_numpy(dtype=float)
    mechanism_attribution_denominator = mechanism_attribution_baseline_error[mechanism_attribution_indices].mean(axis=1) - tess_error[mechanism_attribution_indices].mean(axis=1)

    for mechanism_attribution_method in mechanism_attribution_wide.columns:
        method_error = 1 - mechanism_attribution_wide[mechanism_attribution_method].to_numpy(dtype=float)
        sampled_ler = method_error[mechanism_attribution_indices].mean(axis=1)
        sampled_gap = np.divide(mechanism_attribution_baseline_error[mechanism_attribution_indices].mean(axis=1) - sampled_ler, mechanism_attribution_denominator, out=np.full(len(mechanism_attribution_denominator), np.nan), where=~np.isclose(mechanism_attribution_denominator, 0))
        mechanism_attribution_bootstrap_rows.append({
            'distance': mechanism_attribution_distance,
            'p': mechanism_attribution_p,
            'method': mechanism_attribution_method,
            'LER_ci_low': np.quantile(sampled_ler, 0.025),
            'LER_ci_high': np.quantile(sampled_ler, 0.975),
            'gap_closure_ci_low': np.nanquantile(sampled_gap, 0.025) if np.isfinite(sampled_gap).any() else np.nan,
            'gap_closure_ci_high': np.nanquantile(sampled_gap, 0.975) if np.isfinite(sampled_gap).any() else np.nan,
        })

final_bootstrap = mechanism_attribution_save_csv(pd.DataFrame(mechanism_attribution_bootstrap_rows), mechanism_attribution_data_paths.output('final_paired_bootstrap_csv'))
mechanism_attribution_final_summary


## Figures 1–8


In [ ]:
heat_columns = ['later_BP_correct', 'damped_BP_correct', 'sum_product_correct', 'PCM_any_correct', 'OSD0_correct', 'OSD1_correct', 'full_OSD_correct']
plt.figure(figsize=(9, max(3, len(mechanism_matrix) * 0.12)))
sns.heatmap(mechanism_matrix.set_index('shot_id')[heat_columns], cmap='viridis', cbar_kws={'label': 'logical success'})
plt.title('Figure 1: counterfactual mechanism matrix')
plt.tight_layout()

plt.show()
shown_overlap = overlap.head(20).reset_index(drop=True)
mechanism_attribution_figure, (count_axis, set_axis) = plt.subplots(2, 1, figsize=(11, 6), sharex=True, gridspec_kw={'height_ratios': [2, 3]})
count_axis.bar(np.arange(len(shown_overlap)), shown_overlap.cases, color='steelblue')
count_axis.set_ylabel('gap cases')
count_axis.set_title('Figure 2: non-exclusive mechanism overlap')

for mechanism_attribution_column, pattern in enumerate(shown_overlap.label_pattern):
    active = [row for row, bit in enumerate(pattern) if bit == '1']
    set_axis.scatter(np.full(len(LABELS), mechanism_attribution_column), np.arange(len(LABELS)), color='lightgray', s=20)
    if active:
        set_axis.scatter(np.full(len(active), mechanism_attribution_column), active, color='black', s=28)
        set_axis.plot([mechanism_attribution_column, mechanism_attribution_column], [min(active), max(active)], color='black', linewidth=1)

set_axis.set_yticks(np.arange(len(LABELS)), LABELS)
set_axis.invert_yaxis()
set_axis.set_xlabel('mechanism intersection')
plt.tight_layout()

plt.show()
plt.figure(figsize=(6, 4))
sns.countplot(data=q_min, x='q_min', hue='later_BP_correct')
plt.title('Figure 3: minimum successful OSD order')
plt.tight_layout()

plt.show()
plt.figure(figsize=(6, 4))
sns.histplot(data=pcm_summary, x='PCM_success_fraction', bins=10)
plt.title('Figure 4: equivalent-PCM success fraction')
plt.tight_layout()

plt.show()
if len(pcm_pairs):
    paired_plot = pcm_pairs.melt(id_vars=['shot_id', 'role'], value_vars=['message_instability', 'extrinsic_conflict', 'four_cycles', 'cycle_rank'])
    graph = sns.catplot(
        data=paired_plot,
        x='role',
        y='value',
        col='variable',
        kind='box',
        col_wrap=2,
        sharey=False,
    )
    graph.fig.suptitle('Figure 5: paired failing/successful PCM', y=1.02)

    plt.show()

margin_plot = logical_costs.drop_duplicates('shot_id')
plt.figure(figsize=(7, 4))
sns.boxplot(data=margin_plot, x='cohort', y='logical_margin')
plt.axhline(0, color='black', linewidth=1)
plt.title('Figure 6: exact logical-sector margin')
plt.tight_layout()

plt.show()
repair_heatmap = mechanism_repair[mechanism_repair.split == 'all'].pivot(index='mechanism', columns='intervention', values='repair_rate')
plt.figure(figsize=(8, 5))
sns.heatmap(
    repair_heatmap,
    annot=True,
    vmin=0,
    vmax=1,
    cmap='mako',
)
plt.title('Figure 7: mechanism-conditional repair')
plt.tight_layout()

plt.show()
plot_methods = ['standard_BPOSD', 'Tesseract', 'PCM_ensemble_BP', 'mechanism_aware_BP', 'mechanism_aware_BP_plus_OSD1', 'forced_configured_full_OSD']
plot_data = mechanism_attribution_final_summary[mechanism_attribution_final_summary.method.isin(plot_methods)]
graph = sns.relplot(
    data=plot_data,
    x='p',
    y='LER',
    hue='method',
    col='distance',
    kind='line',
    marker='o',
    facet_kws={'sharey': False},
)
graph.fig.suptitle('Figure 8: paired final LER', y=1.03)

plt.show()


## Completeness and semantic checks


In [ ]:
assert set(LABELS) <= set(mechanism_matrix.columns)
assert set(['random', 'instability', 'ambiguity', 'conflict', 'hybrid']) == set(targeted_rows.strategy)
assert set(['same_PCM_restart', 'random_PCM', 'instability_PCM', 'ambiguity_PCM', 'adaptive_PCM_switching', 'update_rule_correction', 'low_order_OSD', 'baseline']) <= set(targeted_trials.strategy_family)
required_final = {'standard_BP', 'standard_BPOSD', 'Tesseract', 'continued_BP', 'sum_product_BP', 'PCM_ensemble_BP', 'ambiguity_targeted_PCM_BP', 'mechanism_aware_BP', 'mechanism_aware_BP_plus_OSD1', 'forced_configured_full_OSD'}
assert required_final == set(mechanism_attribution_final_trials.method)
assert mechanism_attribution_final_trials.groupby(['distance', 'p', 'method']).shot_id.nunique().groupby(level=[0, 1]).nunique().eq(1).all()
mechanism_attribution_expected_distances = {mechanism_attribution_context_for(level, mechanism_attribution_config['p_values'][0]).distance for level in mechanism_attribution_config['primary_levels'] + mechanism_attribution_config['validation_levels']}
assert set(mechanism_attribution_final_trials.distance) == mechanism_attribution_expected_distances
assert mechanism_attribution_final_trials[mechanism_attribution_final_trials.method == 'mechanism_aware_BP_plus_OSD1'].osd1_used.eq(1).all()
assert logical_costs.groupby('shot_id').sector.nunique().eq(4).all()
assert logical_costs.groupby('shot_id').exact.min().eq(1).all(), 'logical-sector MILP did not prove every optimum'
experiment_frames = [(1, early_stop), (2, osd_orders), (3, pcm_views), (4, pcm_pairs), (5, mechanism_attribution_restart_trials), (6, update_trials), (7, grid_trials), (8, logical_costs), (9, prior_trials), (10, targeted_rows), (11, trapping_pairs), (12, mechanism_matrix), (13, mechanism_repair)]
eligible_counts = {
    4: int(pcm_summary.PCM_any_correct.sum()),
    11: targeted_rows[targeted_rows.logical_correct == 1].shot_id.nunique(),
}
status_rows = []

for number, mechanism_attribution_frame in experiment_frames:
    mechanism_attribution_eligible = eligible_counts.get(number, len(mechanism_attribution_frame))
    execution_status = 'completed' if len(mechanism_attribution_frame) else 'no_eligible_pairs' if number in {4, 11} and mechanism_attribution_eligible == 0 else 'missing'
    status_rows.append({
        'experiment': number,
        'implemented': int(execution_status != 'missing'),
        'execution_status': execution_status,
        'eligible_cases': mechanism_attribution_eligible,
        'trace_rows': len(mechanism_attribution_frame),
    })

mechanism_attribution_experiment_status = pd.DataFrame(status_rows)
assert mechanism_attribution_experiment_status.implemented.eq(1).all(), mechanism_attribution_experiment_status
required_outputs = [mechanism_attribution_population_path, TRAJECTORY_PATH, EARLY_PATH, OSD_PATH, PCM_PATH, PCM_PAIR_PATH, RESTART_PATH, UPDATE_PATH, GRID_PATH, LOGICAL_PATH, PRIOR_PATH, TARGET_PATH, TRAP_PATH, MECHANISM_PATH, TARGETED_PATH, FINAL_PATH, POLICY_PATH, mechanism_attribution_data_paths.output('final_benchmark_csv')]
assert all((path.exists() for path in required_outputs)), [str(path) for path in required_outputs if not path.exists()]
standard_test = mechanism_attribution_final_trials[mechanism_attribution_final_trials.method == 'standard_BPOSD']
test_gap_cases = int(((standard_test.logical_correct == 0) & (standard_test.tesseract_correct == 1)).sum())
mechanism_attribution_save_csv(mechanism_attribution_experiment_status, mechanism_attribution_data_paths.output('experiment_status_csv'))
mechanism_attribution_log(2, {
    'status': 'smoke_passed' if mechanism_attribution_config['smoke_mode'] else 'complete',
    'experiments': mechanism_attribution_experiment_status.experiment.tolist(),
    'gap_cases': len(mechanism_attribution_gap_cases),
    'test_gap_cases': test_gap_cases,
    'test_gap_status': 'observed' if test_gap_cases else 'no_test_gap',
    'final_methods': sorted(required_final),
})
mechanism_attribution_experiment_status
